# OmicsEdgeBio Project 003 — Phase 2C Benchmark Expansion

Frozen Phase 2C design:

- HG003 + HG004
- Illumina + Oxford Nanopore
- 4 benchmark runs
- 50 frozen windows
- 541 assessable segments
- 1,162,571 assessable bases
- 19 chromosomes
- chr20 excluded
- hap.py 0.3.15 + RTG vcfeval 3.12.1

Large genomic inputs remain in ephemeral Google Colab storage.

The notebook embeds the frozen scientific configuration and the audited
Phase 2C runner byte-for-byte. It does not require GitHub credentials or
access to the private repository.

**Safety gate:** `EXECUTE_COMPARISON` defaults to `False`.


## 1. Runtime and execution gate


In [ ]:
import base64
import csv
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

EXECUTE_COMPARISON = False

WORK_ROOT = Path("/content/omicsedge_phase2c")
CONFIG_DIR = WORK_ROOT / "authoritative_config"
RUNNER_DIR = WORK_ROOT / "runner"
RESULTS_DIR = Path("/content/omicsedge_phase2c_results")
ARCHIVE_PATH = Path("/content/omicsedge_phase2c_results.tar.gz")

for directory in (CONFIG_DIR, RUNNER_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def mem_total_bytes():
    for raw in Path("/proc/meminfo").read_text().splitlines():
        if raw.startswith("MemTotal:"):
            return int(raw.split()[1]) * 1024
    return 0

resources = {
    "architecture": platform.machine(),
    "logical_cpus": os.cpu_count() or 0,
    "ram_bytes": mem_total_bytes(),
    "free_bytes": shutil.disk_usage("/content").free,
    "python": sys.version,
}
print(json.dumps(resources, indent=2))

errors = []
if resources["architecture"] not in {"x86_64", "amd64"}:
    errors.append("requires linux/amd64")
if resources["logical_cpus"] < 2:
    errors.append("requires at least 2 logical CPUs")
if resources["ram_bytes"] < 7 * 1024**3:
    errors.append("requires at least 7 GiB RAM")
if resources["free_bytes"] < 15 * 1024**3:
    errors.append("requires at least 15 GiB free temporary disk")
if errors:
    raise RuntimeError("Phase 2C Colab preflight failed: " + "; ".join(errors))

print("EXECUTE_COMPARISON =", EXECUTE_COMPARISON)


## 2. Materialize and verify frozen Project 003 configuration


In [ ]:
EMBEDDED_B64 = {'environment/comparator-lock.yaml': 'bG9ja192ZXJzaW9uOiAiMS4wLjAiCmxvY2tfaWQ6ICJwaGFzZTFhLWhhcHB5LTAuMy4xNS12Y2ZldmFsLTMuMTIuMSIKbG9ja2VkX2F0OiAiMjAyNi0wOS0yOCIKcGxhdGZvcm06CiAgb3M6IGxpbnV4CiAgYXJjaGl0ZWN0dXJlOiBhbWQ2NApoYXBfcHk6CiAgdmVyc2lvbjogIjAuMy4xNSIKICB1cHN0cmVhbV9yZXBvc2l0b3J5OiAiaHR0cHM6Ly9naXRodWIuY29tL0lsbHVtaW5hL2hhcC5weSIKICBnaXRfdGFnOiAidjAuMy4xNSIKICBnaXRfY29tbWl0OiAiODQwMTE2OTViMmZmMjQwNmMxNmEzMzUxMDZkYjY4MzFmYjY3ZmRmZSIKICBzb3VyY2VfYXJjaGl2ZV9zaGEyNTY6ICI4Mzg5MTJmMWFiODA1MjI0MTEwYWYzYzEwMzU4NDlkNTQ4M2I1Y2U0YzU2NWFlMGYwZGI3ZjcyODRhZjA2Njc2IgogIGNvbnRhaW5lcjoKICAgIHJlZmVyZW5jZTogInF1YXkuaW8vYmlvY29udGFpbmVycy9oYXAucHlAc2hhMjU2OmQ2M2I5NjNhNmNiMDFiNDgzMDM5M2IyMjM2OWU3YjkxZDI5OGU0MTU2ZGRlMzUzNzM5ZTc0ZTRjZmE0Zjk2ZDAiCiAgICB0YWdfZm9yX2h1bWFuc19vbmx5OiAiMC4zLjE1LS1weTI3aGNiNzNiM2RfMCIKICAgIG1hbmlmZXN0X2RpZ2VzdDogInNoYTI1NjpkNjNiOTYzYTZjYjAxYjQ4MzAzOTNiMjIzNjllN2I5MWQyOThlNDE1NmRkZTM1MzczOWU3NGU0Y2ZhNGY5NmQwIgogICAgY29tcHJlc3NlZF9sYXllcl9ieXRlczogMjMzNTY4NTE2CnJ0Z190b29sczoKICB2ZXJzaW9uOiAiMy4xMi4xIgogIHVwc3RyZWFtX3JlcG9zaXRvcnk6ICJodHRwczovL2dpdGh1Yi5jb20vUmVhbFRpbWVHZW5vbWljcy9ydGctdG9vbHMiCiAgZ2l0X3RhZzogIjMuMTIuMSIKICBnaXRfY29tbWl0OiAiYTlkMTZkODhiODBmYzY2MGJhM2FkN2Q2YjllN2FjYzQ1ZTBiZWZkNyIKICBsaW51eF94NjRfYXJjaGl2ZV91cmw6ICJodHRwczovL2dpdGh1Yi5jb20vUmVhbFRpbWVHZW5vbWljcy9ydGctdG9vbHMvcmVsZWFzZXMvZG93bmxvYWQvMy4xMi4xL3J0Zy10b29scy0zLjEyLjEtbGludXgteDY0LnppcCIKICBsaW51eF94NjRfYXJjaGl2ZV9ieXRlczogNTA3NDI1MzcKICBsaW51eF94NjRfYXJjaGl2ZV9zaGEyNTY6ICJiYTQzY2FkYmIzZTc5YmZmNzRlN2QyZjg1NDBkZDVlMjAxNTc0ZWM4ZTI5M2JhNTU1ZjNhZWVmZDEzNzVkZmM1Igpjb21wYXJpc29uOgogIGVuZ2luZTogInZjZmV2YWwiCiAgYW5ub3RhdGVkX3ZjZjogdHJ1ZQogIHRocmVhZHM6IDIKICBjb3JlX2ludGVydmFsOgogICAgYXNzZW1ibHk6ICJHUkNoMzgiCiAgICBjb250aWc6ICJjaHIyMCIKICAgIHN0YXJ0XzBiYXNlZDogMTAwMDAwMDAKICAgIGVuZF8wYmFzZWQ6IDExMDAwMDAwCiAgcGFkZGVkX2ludGVydmFsOgogICAgYXNzZW1ibHk6ICJHUkNoMzgiCiAgICBjb250aWc6ICJjaHIyMCIKICAgIHN0YXJ0XzBiYXNlZDogOTk5OTAwMAogICAgZW5kXzBiYXNlZDogMTEwMDEwMDAKICBpbW11dGFibGVfYXJnczoKICAgIC0gIi1WIgogICAgLSAiLS1lbmdpbmU9dmNmZXZhbCIKICAgIC0gIi0tZW5naW5lLXZjZmV2YWwtcGF0aD0vb3B0L3J0Zy10b29scy0zLjEyLjEvcnRnIgogICAgLSAiLS10aHJlYWRzPTIiCnJ1bnRpbWVfY2FwdHVyZV9yZXF1aXJlZDoKICAtIGhhcC5weV92ZXJzaW9uCiAgLSBydGdfdmVyc2lvbgogIC0gamF2YV92ZXJzaW9uCiAgLSBiY2Z0b29sc192ZXJzaW9uCiAgLSBzYW10b29sc192ZXJzaW9uCiAgLSBweXRob25fdmVyc2lvbgogIC0gZGVyaXZlZF9pbWFnZV9kaWdlc3QKICAtIGNvbW1hbmRfYXJndgogIC0gc3Rkb3V0X3NoYTI1NgogIC0gc3RkZXJyX3NoYTI1Ngo=', 'data/phase2/admitted_sources.tsv': 'Y2FuZGlkYXRlX2lkCXNhbXBsZV9pZAl0ZWNobm9sb2d5CXBsYXRmb3JtCWNvdmVyYWdlCXNvdXJjZV9jb3JwdXMJc291cmNlX3N1Ym1pc3Npb25faWQJcGlwZWxpbmVfbmFtZQlyZWZlcmVuY2VfYXNzZW1ibHkJcXVlcnlfdmNmX3VybAlxdWVyeV9zaGEyNTZfdXJsCWV4cGVjdGVkX3F1ZXJ5X3NoYTI1Ngl0cnV0aF92Y2ZfdXJsCWJlbmNobWFya19iZWRfdXJsCXNlbGVjdGlvbl9iYXNpcwlwZXJmb3JtYW5jZV9jb25zdWx0ZWRfZm9yX3NlbGVjdGlvbglwaGFzZTJhX2FkbWlzc2lvbl9zdGF0ZQpwaGFzZTItaGcwMDMtNjB6NTkJSEcwMDMJSUxMVU1JTkEJTm92YVNlcSA2MDAwCTM1eAlwcmVjaXNpb25GREEgVHJ1dGggQ2hhbGxlbmdlIFYyIC8gTklTVCBQRFIgbWRzMi0yMzM2CTYwWjU5CVJOLUlsbHVtaW5hCUdSQ2gzOAlodHRwczovL2RhdGEubmlzdC5nb3Yvb2QvZHMvYXJrOi84ODQzNC9tZHMyLTIzMzYvc3VibWlzc2lvbl92Y2ZzLzYwWjU5LzYwWjU5X0hHMDAzLnZjZi5neglodHRwczovL2RhdGEubmlzdC5nb3Yvb2QvZHMvYXJrOi84ODQzNC9tZHMyLTIzMzYvc3VibWlzc2lvbl92Y2ZzLzYwWjU5LzYwWjU5X0hHMDAzLnZjZi5nei5zaGEyNTYJYzQzODRiNzM0YWU3ZjU0Y2VlZGMzYzVlNWI1MmVmNGI0ODA1Y2JhOGRjYzcwMDc5YTVkY2Y0MTIwMTBhYjQ0YQlodHRwczovL2Z0cC10cmFjZS5uY2JpLm5sbS5uaWguZ292L1JlZmVyZW5jZVNhbXBsZXMvZ2lhYi9yZWxlYXNlL0FzaGtlbmF6aW1UcmlvL0hHMDAzX05BMjQxNDlfZmF0aGVyL05JU1R2NC4yLjEvR1JDaDM4L0hHMDAzX0dSQ2gzOF8xXzIyX3Y0LjIuMV9iZW5jaG1hcmsudmNmLmd6CWh0dHBzOi8vZnRwLXRyYWNlLm5jYmkubmxtLm5paC5nb3YvUmVmZXJlbmNlU2FtcGxlcy9naWFiL3JlbGVhc2UvQXNoa2VuYXppbVRyaW8vSEcwMDNfTkEyNDE0OV9mYXRoZXIvTklTVHY0LjIuMS9HUkNoMzgvSEcwMDNfR1JDaDM4XzFfMjJfdjQuMi4xX2JlbmNobWFya19ub2luY29uc2lzdGVudC5iZWQJU2FtZSBQaGFzZSAxIElsbHVtaW5hIHN1Ym1pc3Npb24gZmFtaWx5OyBwdWJsaWMgdHJpbyBhcnRpZmFjdCBzZWxlY3RlZCB3aXRob3V0IGNvbnN1bHRpbmcgYmVuY2htYXJrIHBlcmZvcm1hbmNlCUZhbHNlCUVMSUdJQkxFCnBoYXNlMi1oZzAwMy1ydTg4bglIRzAwMwlPTlQJUHJvbWV0aElPTiBSOS40CTg1eAlwcmVjaXNpb25GREEgVHJ1dGggQ2hhbGxlbmdlIFYyIC8gTklTVCBQRFIgbWRzMi0yMzM2CVJVODhOCVBFUFBFUi1EZWVwVmFyaWFudCAoT05UKQlHUkNoMzgJaHR0cHM6Ly9kYXRhLm5pc3QuZ292L29kL2RzL2FyazovODg0MzQvbWRzMi0yMzM2L3N1Ym1pc3Npb25fdmNmcy9SVTg4Ti9SVTg4Tl9IRzAwMy52Y2YuZ3oJaHR0cHM6Ly9kYXRhLm5pc3QuZ292L29kL2RzL2FyazovODg0MzQvbWRzMi0yMzM2L3N1Ym1pc3Npb25fdmNmcy9SVTg4Ti9SVTg4Tl9IRzAwMy52Y2YuZ3ouc2hhMjU2CTg0YWM1YzUyNmE5MjU0OGZhNmYxY2U0NjEwODY3NGMyOWQzMmVhNzU0Zjg2M2M5ZTkwZmJjYTQxYTQ0ZWRjMjAJaHR0cHM6Ly9mdHAtdHJhY2UubmNiaS5ubG0ubmloLmdvdi9SZWZlcmVuY2VTYW1wbGVzL2dpYWIvcmVsZWFzZS9Bc2hrZW5hemltVHJpby9IRzAwM19OQTI0MTQ5X2ZhdGhlci9OSVNUdjQuMi4xL0dSQ2gzOC9IRzAwM19HUkNoMzhfMV8yMl92NC4yLjFfYmVuY2htYXJrLnZjZi5neglodHRwczovL2Z0cC10cmFjZS5uY2JpLm5sbS5uaWguZ292L1JlZmVyZW5jZVNhbXBsZXMvZ2lhYi9yZWxlYXNlL0FzaGtlbmF6aW1UcmlvL0hHMDAzX05BMjQxNDlfZmF0aGVyL05JU1R2NC4yLjEvR1JDaDM4L0hHMDAzX0dSQ2gzOF8xXzIyX3Y0LjIuMV9iZW5jaG1hcmtfbm9pbmNvbnNpc3RlbnQuYmVkCVNhbWUgUGhhc2UgMSBPTlQgc3VibWlzc2lvbiBmYW1pbHk7IHB1YmxpYyB0cmlvIGFydGlmYWN0IHNlbGVjdGVkIHdpdGhvdXQgY29uc3VsdGluZyBiZW5jaG1hcmsgcGVyZm9ybWFuY2UJRmFsc2UJRUxJR0lCTEUKcGhhc2UyLWhnMDA0LTYwejU5CUhHMDA0CUlMTFVNSU5BCU5vdmFTZXEgNjAwMAkzNXgJcHJlY2lzaW9uRkRBIFRydXRoIENoYWxsZW5nZSBWMiAvIE5JU1QgUERSIG1kczItMjMzNgk2MFo1OQlSTi1JbGx1bWluYQlHUkNoMzgJaHR0cHM6Ly9kYXRhLm5pc3QuZ292L29kL2RzL2FyazovODg0MzQvbWRzMi0yMzM2L3N1Ym1pc3Npb25fdmNmcy82MFo1OS82MFo1OV9IRzAwNC52Y2YuZ3oJaHR0cHM6Ly9kYXRhLm5pc3QuZ292L29kL2RzL2FyazovODg0MzQvbWRzMi0yMzM2L3N1Ym1pc3Npb25fdmNmcy82MFo1OS82MFo1OV9IRzAwNC52Y2YuZ3ouc2hhMjU2CWI0MTg0YzE2NGVmMmNlZjYwYjUwZmE5ZGI1NzljMjU0OGVmYTVkNDMwMWQxMDVhNDlmOGQ3ODAzMTE0ZTFlZmYJaHR0cHM6Ly9mdHAtdHJhY2UubmNiaS5ubG0ubmloLmdvdi9SZWZlcmVuY2VTYW1wbGVzL2dpYWIvcmVsZWFzZS9Bc2hrZW5hemltVHJpby9IRzAwNF9OQTI0MTQzX21vdGhlci9OSVNUdjQuMi4xL0dSQ2gzOC9IRzAwNF9HUkNoMzhfMV8yMl92NC4yLjFfYmVuY2htYXJrLnZjZi5neglodHRwczovL2Z0cC10cmFjZS5uY2JpLm5sbS5uaWguZ292L1JlZmVyZW5jZVNhbXBsZXMvZ2lhYi9yZWxlYXNlL0FzaGtlbmF6aW1UcmlvL0hHMDA0X05BMjQxNDNfbW90aGVyL05JU1R2NC4yLjEvR1JDaDM4L0hHMDA0X0dSQ2gzOF8xXzIyX3Y0LjIuMV9iZW5jaG1hcmtfbm9pbmNvbnNpc3RlbnQuYmVkCVNhbWUgUGhhc2UgMSBJbGx1bWluYSBzdWJtaXNzaW9uIGZhbWlseTsgcHVibGljIHRyaW8gYXJ0aWZhY3Qgc2VsZWN0ZWQgd2l0aG91dCBjb25zdWx0aW5nIGJlbmNobWFyayBwZXJmb3JtYW5jZQlGYWxzZQlFTElHSUJMRQpwaGFzZTItaGcwMDQtcnU4OG4JSEcwMDQJT05UCVByb21ldGhJT04gUjkuNAk4NXgJcHJlY2lzaW9uRkRBIFRydXRoIENoYWxsZW5nZSBWMiAvIE5JU1QgUERSIG1kczItMjMzNglSVTg4TglQRVBQRVItRGVlcFZhcmlhbnQgKE9OVCkJR1JDaDM4CWh0dHBzOi8vZGF0YS5uaXN0Lmdvdi9vZC9kcy9hcms6Lzg4NDM0L21kczItMjMzNi9zdWJtaXNzaW9uX3ZjZnMvUlU4OE4vUlU4OE5fSEcwMDQudmNmLmd6CWh0dHBzOi8vZGF0YS5uaXN0Lmdvdi9vZC9kcy9hcms6Lzg4NDM0L21kczItMjMzNi9zdWJtaXNzaW9uX3ZjZnMvUlU4OE4vUlU4OE5fSEcwMDQudmNmLmd6LnNoYTI1NgllYzBlNDgxZGRlMDQ4NmQzMGM5Zjg1YzkzMjNjMWJmNzUwMDhlMmJhZDJiYTUwZTJhYzQ0ZTdlMTUyNzdkYWJkCWh0dHBzOi8vZnRwLXRyYWNlLm5jYmkubmxtLm5paC5nb3YvUmVmZXJlbmNlU2FtcGxlcy9naWFiL3JlbGVhc2UvQXNoa2VuYXppbVRyaW8vSEcwMDRfTkEyNDE0M19tb3RoZXIvTklTVHY0LjIuMS9HUkNoMzgvSEcwMDRfR1JDaDM4XzFfMjJfdjQuMi4xX2JlbmNobWFyay52Y2YuZ3oJaHR0cHM6Ly9mdHAtdHJhY2UubmNiaS5ubG0ubmloLmdvdi9SZWZlcmVuY2VTYW1wbGVzL2dpYWIvcmVsZWFzZS9Bc2hrZW5hemltVHJpby9IRzAwNF9OQTI0MTQzX21vdGhlci9OSVNUdjQuMi4xL0dSQ2gzOC9IRzAwNF9HUkNoMzhfMV8yMl92NC4yLjFfYmVuY2htYXJrX25vaW5jb25zaXN0ZW50LmJlZAlTYW1lIFBoYXNlIDEgT05UIHN1Ym1pc3Npb24gZmFtaWx5OyBwdWJsaWMgdHJpbyBhcnRpZmFjdCBzZWxlY3RlZCB3aXRob3V0IGNvbnN1bHRpbmcgYmVuY2htYXJrIHBlcmZvcm1hbmNlCUZhbHNlCUVMSUdJQkxFCg==', 'data/phase2/selected_assessable_segments.bed': 'Y2hyMQk2NjUxNzQxCTY2NTE5NjYKY2hyMQk2NjUxOTg4CTY2NTYwNDkKY2hyMQk2NjU2MTk2CTY2NTY2ODAKY2hyMQk2NjU2Nzg0CTY2NTc4MDYKY2hyMQk2NjU3ODIzCTY2NTk0NzEKY2hyMQk2NjYwMTAxCTY2NjIwNjcKY2hyMQk2NjYyMTg2CTY2NjQyMzYKY2hyMQk2NjY0MjYyCTY2NjYzMzQKY2hyMQk2NjY2NDQ0CTY2NzExODkKY2hyMQk2NjcyMTA1CTY2NzMxNDQKY2hyMQk2NjczMjQ2CTY2NzY3NDEKY2hyMQkzNTM2MTA5MAkzNTM2NTIzNApjaHIxCTM1MzY1MjY4CTM1MzY4MDEwCmNocjEJMzUzNjgxMjMJMzUzNzAyODcKY2hyMQkzNTM3MDM4OQkzNTM3MDY3NgpjaHIxCTM1MzcwNzgzCTM1MzcxMDExCmNocjEJMzUzNzExNTkJMzUzNzM1MzkKY2hyMQkzNTM3MzU3MQkzNTM3Mzk2OApjaHIxCTM1Mzc0MDcwCTM1Mzg0NzkwCmNocjEJMzUzODQ4OTIJMzUzODYwOTAKY2hyMQkzNjk5NzA5NwkzNzAwMDM5OQpjaHIxCTM3MDAwNTI1CTM3MDAyNjE5CmNocjEJMzcwMDI3MjIJMzcwMDI5NzAKY2hyMQkzNzAwMzEwOAkzNzAxMDY3NQpjaHIxCTM3MDEwNzc3CTM3MDE0NzgzCmNocjEJMzcwMTQ5ODAJMzcwMTczMTgKY2hyMQkzNzAxNzQ1MwkzNzAyMTA3MQpjaHIxCTM3MDIxMTc4CTM3MDIyMDk3CmNocjIJNjUxMTA2OTEJNjUxMTIwNzcKY2hyMgk2NTExMjE5OAk2NTExNDgxMwpjaHIyCTY1MTE0OTE2CTY1MTE3NjgyCmNocjIJNjUxMTc3ODUJNjUxMTk3ODkKY2hyMgk2NTExOTk0MAk2NTEyMDQwNApjaHIyCTY1MTIwNTA3CTY1MTIxMTg0CmNocjIJNjUxMjEyODYJNjUxMjIxMDMKY2hyMgk2NTEyMjIxOAk2NTEyMjM4NwpjaHIyCTY1MTIyNDEzCTY1MTIyOTIzCmNocjIJNjUxMjMyMjEJNjUxMjUzNzIKY2hyMgk2NTEyNTQ5NAk2NTEyNTc5NgpjaHIyCTY1MTI1OTE4CTY1MTMwNTk1CmNocjIJNjUxMzA2MjEJNjUxMzIwMDkKY2hyMgk2NTEzMjE0MQk2NTEzNTY5MQpjaHIyCTEzNDk0OTA4MwkxMzQ5NDk3NTEKY2hyMgkxMzQ5NDk4NjQJMTM0OTYyMjgxCmNocjIJMTM0OTYyNDEyCTEzNDk2NjQxMwpjaHIyCTEzNDk2NjQzMQkxMzQ5Njc4MDYKY2hyMgkxMzQ5Njc5MDkJMTM0OTY4MzY5CmNocjIJMTM0OTY4OTY5CTEzNDk2OTE4MApjaHIyCTEzNDk2OTE5OQkxMzQ5NzA1OTAKY2hyMgkxMzQ5NzA2MjEJMTM0OTc0MDgzCmNocjIJMTg1NjE5NzUzCTE4NTYyMTA5MgpjaHIyCTE4NTYyMTE5OAkxODU2MjYyNjkKY2hyMgkxODU2MjY0MjIJMTg1NjI2NTIwCmNocjIJMTg1NjI2NTQ3CTE4NTYyNzE4MApjaHIyCTE4NTYyNzI4MgkxODU2Mjc2NDQKY2hyMgkxODU2Mjc3NDYJMTg1NjI5NDgyCmNocjIJMTg1NjI5NjEwCTE4NTY0MTU3NQpjaHIyCTE4NTY0MTkyOAkxODU2NDQ3NTMKY2hyMgkyMjkwOTE1ODYJMjI5MDkxOTMyCmNocjIJMjI5MDkyMzYwCTIyOTA5MzA4MwpjaHIyCTIyOTA5MzE4OQkyMjkwOTM3MTcKY2hyMgkyMjkwOTM4MjAJMjI5MTAzNTIwCmNocjIJMjI5MTAzNjIzCTIyOTEwNzQ1MgpjaHIyCTIyOTEwNzU1NQkyMjkxMTUzNTEKY2hyMgkyMjkxMTU0ODgJMjI5MTE2MDYyCmNocjIJMjI5MTE2MDgyCTIyOTExNjU4NgpjaHIzCTE4MDMwNzA0OAkxODAzMTg4NDYKY2hyMwkxODAzMTg4ODAJMTgwMzIyMDg4CmNocjMJMTgwMzIyMTk4CTE4MDMyMjg1MApjaHIzCTE4MDMyMjg3NQkxODAzMjM0OTMKY2hyMwkxODAzMjM2MjIJMTgwMzI2MDQ1CmNocjMJMTgwMzI2MDYzCTE4MDMyNjM4NQpjaHIzCTE4MDMyNjQ0NwkxODAzMjY3MTQKY2hyMwkxODAzMjY3NDcJMTgwMzMyMDQ4CmNocjQJNDgzMjk2NjEJNDgzNDI2NjAKY2hyNAk0ODM0Mjc4Nwk0ODM0Njc4NApjaHI0CTQ4MzQ2ODg3CTQ4MzQ4NjM4CmNocjQJNDgzNDg3NDAJNDgzNTIwNjEKY2hyNAk0ODM1MjE2Ngk0ODM1NDY2MQpjaHI0CTc1ODY2OTEyCTc1ODY4MjY0CmNocjQJNzU4NjgzODQJNzU4NzA4MjcKY2hyNAk3NTg3MDkzOQk3NTg3NDA4MApjaHI0CTc1ODc0MTgyCTc1ODc2OTM2CmNocjQJNzU4NzcxMTEJNzU4NzcyODYKY2hyNAk3NTg3NzQ2MAk3NTg3OTUyNQpjaHI0CTc1ODc5NjI3CTc1ODc5Nzg3CmNocjQJNzU4Nzk4ODkJNzU4ODA3NjcKY2hyNAk3NTg4MDg2OQk3NTg4MzUxNwpjaHI0CTc1ODg0MTY1CTc1ODg0MzczCmNocjQJNzU4ODQ1MjkJNzU4ODU1ODgKY2hyNAk3NTg4NTg2OAk3NTg4NTk1OApjaHI0CTc1ODg3MDAwCTc1ODg3NTY5CmNocjQJNzU4ODc2NzQJNzU4OTE5MTIKY2hyNAk3NjUxOTQ0Ngk3NjUyMjEwMApjaHI0CTc2NTIyMjE2CTc2NTI4MjczCmNocjQJNzY1Mjg0MjYJNzY1Mjg0OTMKY2hyNAk3NjUyODU5OQk3NjUzMTg1NgpjaHI0CTc2NTMyMTE2CTc2NTQ0MzU2CmNocjQJMTgwMDM1MjEyCTE4MDA0MTE4NApjaHI0CTE4MDA0MTIxMQkxODAwNDM1MzcKY2hyNAkxODAwNDM2MzkJMTgwMDQ1NjE2CmNocjQJMTgwMDQ1NzI1CTE4MDA1ODY3NQpjaHI0CTE4MDA1ODc3NwkxODAwNjAyMTIKY2hyNAkxODkyODE3MzIJMTg5MjgyNzg0CmNocjQJMTg5MjgyODg2CTE4OTI4MzIwNApjaHI0CTE4OTI4MzIyMQkxODkyODMzNjcKY2hyNAkxODkyODMzODkJMTg5MjgzODgzCmNocjQJMTg5MjgzOTkzCTE4OTI4NDA0OApjaHI0CTE4OTI4NDA2NQkxODkyODQzNDIKY2hyNAkxODkyODQ1MDMJMTg5Mjg3MDgyCmNocjQJMTg5Mjg3MTY1CTE4OTI4Nzg5NgpjaHI0CTE4OTI4NzkxMwkxODkyODg3MjUKY2hyNAkxODkyODg3NzMJMTg5Mjg5ODE5CmNocjQJMTg5Mjg5OTIxCTE4OTI5MDU5NwpjaHI0CTE4OTI5MDcwMAkxODkyOTA5MDYKY2hyNAkxODkyOTEwMjAJMTg5Mjk2Mzg0CmNocjQJMTg5Mjk2NDg2CTE4OTMwMTEwMQpjaHI1CTEwMzEzNjU1CTEwMzE1NTkyCmNocjUJMTAzMTU2OTQJMTAzMTcwNTAKY2hyNQkxMDMxNzU2OQkxMDMyNzA5NgpjaHI1CTEwMzI3MTk5CTEwMzMyNzQ3CmNocjUJMTAzMzI4NDkJMTAzMzY0NTYKY2hyNQkxMDMzNjU1OQkxMDMzODY1NQpjaHI1CTEzNzU0NDMyMgkxMzc1NTA1MTYKY2hyNQkxMzc1NTA1NTgJMTM3NTU2Njc3CmNocjUJMTM3NTU3Mjk2CTEzNzU1NzM3MQpjaHI1CTEzNzU1NzQ3NAkxMzc1Njg2MTMKY2hyNQkxMzc1Njg3NTMJMTM3NTY5MzIyCmNocjUJMTQxMDg5OTI5CTE0MTA5MDM4NQpjaHI1CTE0MTA5MDQ4OAkxNDEwOTI2MjgKY2hyNQkxNDEwOTI3MzAJMTQxMDkyOTMwCmNocjUJMTQxMDkzMDIxCTE0MTA5MzE2MgpjaHI1CTE0MTA5MzE4MAkxNDEwOTQyODMKY2hyNQkxNDEwOTQzODYJMTQxMDk1MDAwCmNocjUJMTQxMDk1MDE4CTE0MTA5NzMzMgpjaHI1CTE0MTA5NzM0OQkxNDEwOTc2NDUKY2hyNQkxNDEwOTc3NDgJMTQxMDk4MjE0CmNocjUJMTQxMDk4MjYxCTE0MTA5ODQ2MgpjaHI1CTE0MTA5ODQ3OQkxNDEwOTk5OTEKY2hyNQkxNDExMDAwMTIJMTQxMTAwMDc3CmNocjUJMTQxMTAwMTAwCTE0MTEwMzM1NApjaHI1CTE0MTEwMzM3NAkxNDExMDM5NjQKY2hyNQkxNDExMDQwNzAJMTQxMTA0Mzc3CmNocjUJMTQxMTA0Mzk5CTE0MTEwNDYzNQpjaHI1CTE0MTEwNDY1NwkxNDExMDQ5MzIKY2hyNQkxNDExMDQ5NTAJMTQxMTA2NjA3CmNocjUJMTQxMTA2NjI0CTE0MTEwNzg5OQpjaHI1CTE0MTEwODAwNgkxNDExMDgzNzUKY2hyNQkxNDExMDgzOTYJMTQxMTA4NzU3CmNocjUJMTQxMTA4Nzc4CTE0MTExMDc3NgpjaHI1CTE0MTExMDkyNAkxNDExMTEzNjYKY2hyNQkxNDExMTEzODcJMTQxMTEyMTI1CmNocjUJMTQxMTEyMTQyCTE0MTExMjc4MgpjaHI1CTE0MTExMjc5OQkxNDExMTM5NjcKY2hyNQkxNDExMTM5ODQJMTQxMTE0MjY2CmNocjUJMTQxMTE0MzY4CTE0MTExNDkyOQpjaHI2CTUyNzQwNDc3CTUyNzQwOTY1CmNocjYJNTI3NDEwMTIJNTI3NDE0MzMKY2hyNgk1Mjc0MTQ1OAk1Mjc0MTg4NwpjaHI2CTUyNzQxOTA0CTUyNzQ3MjQ0CmNocjYJNTI3NDcyNjEJNTI3NDgwMzcKY2hyNgk1Mjc0ODA1NQk1Mjc0OTU0MgpjaHI2CTUyNzQ5NTYyCTUyNzUyODczCmNocjYJNTI3NTI4OTQJNTI3NTUxNjQKY2hyNgk1Mjc1NTI2Nwk1Mjc1NzI0NQpjaHI2CTUyNzU3MjY4CTUyNzU5NTEyCmNocjYJNTI3NTk2MjQJNTI3NjQ4NTQKY2hyNgk1Mjc2NDg3NQk1Mjc2NTQ3NwpjaHI2CTc3MzE3Mzk3CTc3MzE3NDM5CmNocjYJNzczMTc1NDMJNzczMjE5NzAKY2hyNgk3NzMyMjEwMgk3NzM0MjM5NwpjaHI2CTE2NzMyNDU0MwkxNjczMzM4MjYKY2hyNgkxNjczMzM4NDMJMTY3MzM1MTE5CmNocjYJMTY3MzM1MTM2CTE2NzMzNTgxNgpjaHI2CTE2NzMzNTg0NAkxNjczMzkyNTIKY2hyNgkxNjczMzkzNTQJMTY3MzQyNzk2CmNocjYJMTY3MzQyOTA5CTE2NzM0ODczMApjaHI2CTE2NzM0ODgzNgkxNjczNDk0OTAKY2hyNwk1MTE0NjY0CTUxMTUwNzkKY2hyNwk1MTE1MjI4CTUxMTU1ODAKY2hyNwk1MTE1NjgzCTUxMTk1NjcKY2hyNwk1MTE5NjcxCTUxMjAyMDMKY2hyNwk1MTIwMjI0CTUxMjAzNDUKY2hyNwk1MTIwMzY5CTUxMjA1MTIKY2hyNwk1MTIwNTMzCTUxMjA2MjgKY2hyNwk1MTIwNzMxCTUxMjE0NzAKY2hyNwk1MTIxNTcyCTUxMjIwODQKY2hyNwk1MTIyMTAxCTUxMjIxNzYKY2hyNwk1MTIyMjc5CTUxMjM4OTQKY2hyNwk1MTI0NTgyCTUxMjQ3MDEKY2hyNwk1MTI0NzMzCTUxMjUzNTcKY2hyNwk1MTI1NDYwCTUxMjU1NDQKY2hyNwk1MTI1NjQ3CTUxMjY3MzUKY2hyNwk1MTI2NzU2CTUxMjcyODAKY2hyNwk1MTI3Mjk3CTUxMjc4NzMKY2hyNwk1MTI3OTc2CTUxMjgxNjMKY2hyNwk1MTI4MjczCTUxMjg1OTQKY2hyNwk1MTI4NjExCTUxMjg4NTkKY2hyNwk1MTI5MTI2CTUxMjk1MTQKY2hyNwk1MTI5NjI3CTUxMzAyNjYKY2hyNwk1MTMwNTAxCTUxMzA2NzkKY2hyNwk1MTMwNzgyCTUxMzExMjAKY2hyNwk1MTMxMjMwCTUxMzE4NzgKY2hyNwk1MTMyMTQ0CTUxMzI2ODcKY2hyNwk1MTMyNzE0CTUxMzI4MTUKY2hyNwk1MTMzMTQ0CTUxMzQwMzUKY2hyNwk1MTM0MDU3CTUxMzQxNzAKY2hyNwk1MTM3MDY0CTUxMzc4MTgKY2hyNwk1MTM3OTIxCTUxMzgxNjgKY2hyNwk1MTM4MzM1CTUxMzg3MDMKY2hyNwk1MTM4NzI0CTUxMzg4MTgKY2hyNwk1MTM4OTY4CTUxMzkwMTkKY2hyNwk1MTM5MDM2CTUxMzkxNzUKY2hyNwk1MTM5MjE4CTUxMzk1NDYKY2hyNwkxNTY1NzY2NAkxNTY1OTc1MwpjaHI3CTE1NjU5Nzg2CTE1NjYwOTYxCmNocjcJMTU2NjEwNzMJMTU2NjIwNjIKY2hyNwkxNTY2MjA4NAkxNTY2NjcwMApjaHI3CTE1NjY2ODM4CTE1NjY3MjM4CmNocjcJMTU2NjczNDAJMTU2NzI4NDEKY2hyNwkxNTY3Mjk0OAkxNTY3MzU0NgpjaHI3CTE1NjczNjYzCTE1NjgyNjY0CmNocjcJODc4OTUyODAJODc4OTg3ODUKY2hyNwk4Nzg5ODgxMwk4NzkwMzYzOApjaHI3CTg3OTAzNzQxCTg3OTA1OTQ5CmNocjcJODc5MDYwNTEJODc5MDczNDkKY2hyNwk4NzkwNzM2Ngk4NzkxMDAxMApjaHI3CTg3OTEwMTEzCTg3OTE0NDE1CmNocjcJODc5MTQ1MTcJODc5MjAyODAKY2hyNwk5ODM0NTk0OAk5ODM0ODUwMwpjaHI3CTk4MzQ4NjM0CTk4MzUzOTEwCmNocjcJOTgzNTQwMjcJOTgzNTU1MDUKY2hyNwk5ODM1NTYyMwk5ODM1NjYxNApjaHI3CTk4MzU2NzE4CTk4MzU2ODUwCmNocjcJOTgzNTcyMTEJOTgzNTc1MTQKY2hyNwk5ODM1NzYxNgk5ODM1OTI2OApjaHI3CTk4MzU5Mjk2CTk4MzU5Njk0CmNocjcJOTgzNTk4MjEJOTgzNTk4NzQKY2hyNwk5ODM1OTk3Ngk5ODM2Mjk3NgpjaHI3CTk4MzYzMDg2CTk4MzY0OTM1CmNocjcJOTgzNjUxMDAJOTgzNjczNjkKY2hyNwk5ODM2NzY5Nwk5ODM2ODQxMwpjaHI3CTk4MzY4NDQ4CTk4MzY4NzU0CmNocjcJOTgzNjg4NjMJOTgzNjk2MTIKY2hyNwk5ODM2OTcxOAk5ODM3MDMyMQpjaHI3CTk4MzcwNDIzCTk4MzcwOTQ4CmNocjgJMzk4MDY4NwkzOTgxNDQ5CmNocjgJMzk4MTU3OAkzOTgyMTA5CmNocjgJMzk4MjI1MAkzOTgzMDg0CmNocjgJMzk4MzE4NgkzOTgzNzQzCmNocjgJMzk4NDE3MwkzOTg0NjQ5CmNocjgJMzk4NDgxOQkzOTk0Mzk2CmNocjgJMzk5NDUwMAkzOTk2MzkzCmNocjgJMzk5NjQ5NQkzOTk3Nzk2CmNocjgJMzk5Nzg5OQk0MDAwMDgyCmNocjgJNDAwMDM0Ngk0MDA1Njg3CmNocjgJMjQ5MDU3NDAJMjQ5MTgxNjYKY2hyOAkyNDkxODIyMwkyNDkyMTM5MgpjaHI4CTI0OTIxNzU0CTI0OTI0NjYyCmNocjgJMjQ5MjQ3NjUJMjQ5MzA3NDAKY2hyOAk2NDg5NjE2NQk2NDkxMTU2OApjaHI4CTY0OTExNTg1CTY0OTIwMDQ1CmNocjgJNjQ5MjAxNDcJNjQ5MjExNjUKY2hyOQk5OTk4MTk5Mgk5OTk4MjM2NQpjaHI5CTk5OTgyNDY3CTk5OTgzNTI1CmNocjkJOTk5ODM2MjcJOTk5OTA4MzYKY2hyOQk5OTk5MDg1NQk5OTk5MTY5OQpjaHI5CTk5OTkxNzE2CTk5OTk1ODY5CmNocjkJOTk5OTU5OTMJOTk5OTg3MjYKY2hyOQk5OTk5ODc0Nwk5OTk5OTEwOApjaHI5CTk5OTk5MjM1CTEwMDAwNjk5MgpjaHIxMAk2ODU3ODcyOAk2ODU4MDI2MgpjaHIxMAk2ODU4MDM2NQk2ODU4MDczMQpjaHIxMAk2ODU4MDg1Ngk2ODU4NDIwMQpjaHIxMAk2ODU4NDIyNgk2ODU4NTcwNwpjaHIxMAk2ODU4NTczMgk2ODU4NTk1MQpjaHIxMAk2ODU4NjA3MQk2ODU4NjQzNApjaHIxMAk2ODU4NjY5Mwk2ODU4Nzg3MQpjaHIxMAk2ODU4NzkwMwk2ODU4ODE1NQpjaHIxMAk2ODU4ODE3Ngk2ODU4ODY2MQpjaHIxMAk2ODU4ODY4Mwk2ODU4ODc3MQpjaHIxMAk2ODU4ODg3NAk2ODU4OTA2NQpjaHIxMAk2ODU4OTE3Ngk2ODU4OTQzMgpjaHIxMAk2ODU4OTQ4OAk2ODU4OTYxMQpjaHIxMAk2ODU4OTcyMAk2ODU5MDEwMgpjaHIxMAk2ODU5MDEzMQk2ODU5MDI3OApjaHIxMAk2ODU5MDMyMAk2ODU5MDc4NApjaHIxMAk2ODU5MDg4Nwk2ODU5MzIyNQpjaHIxMAk2ODU5MzI2MQk2ODU5Mzg2NApjaHIxMAk2ODU5Mzk2OAk2ODU5NTU2MQpjaHIxMAk2ODU5NTY2NQk2ODU5NTg2OApjaHIxMAk2ODU5NjA3OAk2ODU5Njk4NQpjaHIxMAk2ODU5NzA1Ngk2ODU5ODYyOApjaHIxMAk2ODU5ODc1MQk2ODYwMzcyOApjaHIxMAk4MzEyMzgyNgk4MzEyNTgzMwpjaHIxMAk4MzEyNTkzNQk4MzEzNjQzMApjaHIxMAk4MzEzNjUzMwk4MzE0NTI4NQpjaHIxMAk4MzE0NTM4OAk4MzE0ODgyNgpjaHIxMgk2ODA0OTU2Nwk2ODA1NDI1MApjaHIxMgk2ODA1NDMwMQk2ODA1NDM3NApjaHIxMgk2ODA1NDQ4Mwk2ODA3MDAzNQpjaHIxMgk2ODA3MDA2MAk2ODA3NDU2NwpjaHIxMgkxMjg5OTk1MjgJMTI5MDAwOTc0CmNocjEyCTEyOTAwMTE0MgkxMjkwMDQyNzIKY2hyMTIJMTI5MDA2Mjk3CTEyOTAxMTgxMQpjaHIxMgkxMjkwMTE5MzAJMTI5MDEyOTQ5CmNocjEyCTEyOTAxMzA1MQkxMjkwMTMyNjcKY2hyMTIJMTI5MDEzMzg3CTEyOTAxNDAxOQpjaHIxMgkxMjkwMTQxODkJMTI5MDE0Mjc0CmNocjEyCTEyOTAxNDI5MQkxMjkwMTc2MDYKY2hyMTIJMTI5MDE3NzA5CTEyOTAxNzc2NwpjaHIxMgkxMjkwMTg1MzgJMTI5MDIxOTY1CmNocjEyCTEyOTAyMjA4NgkxMjkwMjQ1MjgKY2hyMTMJNDYyNTg2MTIJNDYyNjQ2MTEKY2hyMTMJNDYyNjQ2MzcJNDYyNjkxNjEKY2hyMTMJNDYyNjkyNjMJNDYyNjk3NzkKY2hyMTMJNDYyNjk5NTcJNDYyNzEwMDAKY2hyMTMJNDYyNzgwMDAJNDYyODM2MTIKY2hyMTQJNDExMDQzODQJNDExMDYzOTYKY2hyMTQJNDExMDY1MDQJNDExMTA3MzQKY2hyMTQJNDExMTA4NTkJNDExMTYwOTQKY2hyMTQJNDExMTYyMzkJNDExMTc5NDcKY2hyMTQJNDExMTgwNjMJNDExMTg0MTMKY2hyMTQJNDExMTg1NDcJNDExMjA4NDAKY2hyMTQJNDExMjA5NTIJNDExMjg1MDMKY2hyMTQJNDExMjg2MDYJNDExMjkwMzEKY2hyMTQJNDExMjkxNjUJNDExMjkzODQKY2hyMTQJNjM1NDkzMTAJNjM1NTA2NDgKY2hyMTQJNjM1NTEyOTYJNjM1NTgyNTMKY2hyMTQJNjM1NTgzOTYJNjM1NTk4NDkKY2hyMTQJNjM1NTk4OTgJNjM1NjExNDYKY2hyMTQJNjM1NjExNzIJNjM1NjEzNDgKY2hyMTQJNjM1NjE0NTEJNjM1NjIxNDUKY2hyMTQJNjM1NjIyNDgJNjM1Njk1MzQKY2hyMTQJNjM1Njk2MzkJNjM1Njk4OTcKY2hyMTQJNjM1NzA0OTcJNjM1NzM5NDIKY2hyMTQJNjM1NzQwODAJNjM1NzQzMTAKY2hyMTQJODE2NTI4NzQJODE2NjA4MzAKY2hyMTQJODE2NjA5MzgJODE2NjM2MTgKY2hyMTQJODE2NjM3MjAJODE2NjQxMzUKY2hyMTQJODE2NjQyNjQJODE2NjQzMzYKY2hyMTQJODE2NjQ0NjgJODE2NzI1NjQKY2hyMTQJODE2NzI1OTQJODE2Nzc4NzQKY2hyMTUJNjY5ODgwNjAJNjY5OTE0MjkKY2hyMTUJNjY5OTE1OTUJNjY5OTMyNDYKY2hyMTUJNjY5OTM0MzIJNjY5OTQ3OTYKY2hyMTUJNjY5OTQ5NDUJNjY5OTc4MDEKY2hyMTUJNjY5OTc4MTgJNjY5OTg5ODkKY2hyMTUJNjY5OTkwMDYJNjY5OTkxNTUKY2hyMTUJNjY5OTkyNTkJNjY5OTk2MzAKY2hyMTUJNjY5OTk2NDcJNjY5OTk5MzkKY2hyMTUJNjcwMDAwNDEJNjcwMDA2NTUKY2hyMTUJNjcwMDA3NTcJNjcwMDE3NjIKY2hyMTUJNjcwMDE5MDYJNjcwMDIxNjgKY2hyMTUJNjcwMDIyNzAJNjcwMDIzNTcKY2hyMTUJNjcwMDI2NzUJNjcwMDI3MzcKY2hyMTUJNjcwMDI4NDAJNjcwMDM0NzcKY2hyMTUJNjcwMDM1NzkJNjcwMDUzNTYKY2hyMTUJNjcwMDUzODEJNjcwMDc4ODQKY2hyMTUJNjcwMDc5ODYJNjcwMDk0MzgKY2hyMTUJNjcwMDk0NTYJNjcwMTI0NzcKY2hyMTUJNjcwMTI1ODAJNjcwMTMwNjAKY2hyMTUJNjc1MDI4MjMJNjc1MTE2MTUKY2hyMTUJNjc1MTE3MTgJNjc1MTgyMjcKY2hyMTUJNjc1MTgzMzAJNjc1Mjc4MjMKY2hyMTUJNzk5MjgxNjUJNzk5MjgxODQKY2hyMTUJNzk5MjgzMDcJNzk5Mjg1MDIKY2hyMTUJNzk5Mjg2MDQJNzk5Mjk1NTAKY2hyMTUJNzk5Mjk3MDIJNzk5MzAxODMKY2hyMTUJNzk5MzAyODkJNzk5MzY0NDUKY2hyMTUJNzk5MzY1NDkJNzk5Mzc5MDQKY2hyMTUJNzk5MzgwMTUJNzk5NDA5MDYKY2hyMTUJNzk5NDEwMTkJNzk5NDkwOTkKY2hyMTUJNzk5NDkyMDIJNzk5NDk1OTgKY2hyMTUJNzk5NDk3MDEJNzk5NDk5MzIKY2hyMTUJNzk5NTAyNjgJNzk5NTEwODYKY2hyMTUJNzk5NTExODgJNzk5NTEzODIKY2hyMTUJNzk5NTE1MTQJNzk5NTMxNjUKY2hyMTUJODYzNzA0OTEJODYzNzIyNzMKY2hyMTUJODYzNzI1MDMJODYzODAxODAKY2hyMTUJODYzODAzMTUJODYzODU4ODYKY2hyMTUJODYzODYyOTMJODYzOTA2MDkKY2hyMTUJODYzOTA3MTEJODYzOTA5NjYKY2hyMTUJODYzOTEwNjkJODYzOTE1ODkKY2hyMTUJODYzOTE3MTcJODYzOTU0OTEKY2hyMTUJOTE3MTAyNDAJOTE3MTczNzIKY2hyMTUJOTE3MTczOTEJOTE3MTc0ODAKY2hyMTUJOTE3MTc1ODkJOTE3MzAyMzgKY2hyMTUJOTE3MzAzNDEJOTE3MzA0MTgKY2hyMTUJOTE3MzA1MjEJOTE3MzMxMTkKY2hyMTUJOTE3MzMyMjEJOTE3MzUyNDAKY2hyMTUJMTAxNzIxNjk4CTEwMTcyMjE2NgpjaHIxNQkxMDE3MjIyNjkJMTAxNzIyNDU5CmNocjE1CTEwMTcyMjY5MwkxMDE3MjM2ODIKY2hyMTUJMTAxNzIzNzAwCTEwMTcyMzkzNgpjaHIxNQkxMDE3MjM5ODYJMTAxNzI0NTIzCmNocjE1CTEwMTcyNDY4MwkxMDE3MjczNjgKY2hyMTUJMTAxNzI3Mzg1CTEwMTcyODU0OApjaHIxNQkxMDE3Mjg2NTEJMTAxNzI5MDgzCmNocjE1CTEwMTcyOTEwMAkxMDE3MjkzODUKY2hyMTUJMTAxNzI5NDAyCTEwMTcyOTcxMwpjaHIxNQkxMDE3Mjk4MTUJMTAxNzMwMjg3CmNocjE1CTEwMTczMDY0NwkxMDE3MzI2MTEKY2hyMTUJMTAxNzMyNjQxCTEwMTczNDE3NwpjaHIxNQkxMDE3MzQxOTQJMTAxNzM0MjU4CmNocjE1CTEwMTczNDI3NQkxMDE3MzU1MDYKY2hyMTUJMTAxNzM1NjE0CTEwMTczNjE2NQpjaHIxNQkxMDE3MzYyNjcJMTAxNzM2NjI3CmNocjE1CTEwMTczNjcyOQkxMDE3MzY4MDEKY2hyMTUJMTAxNzM2ODI3CTEwMTczNzQ3NgpjaHIxNQkxMDE3Mzc0OTMJMTAxNzM4NjI2CmNocjE1CTEwMTczODY1MQkxMDE3Mzg3OTUKY2hyMTUJMTAxNzM4ODI2CTEwMTczOTAxOApjaHIxNQkxMDE3MzkxMjQJMTAxNzM5NzkxCmNocjE1CTEwMTczOTgxMgkxMDE3Mzk5OTgKY2hyMTUJMTAxNzQwMTQxCTEwMTc0MDU0NgpjaHIxNQkxMDE3NDA2NTAJMTAxNzQxNDEwCmNocjE1CTEwMTc0MTQyNwkxMDE3NDE3MDAKY2hyMTUJMTAxNzQxNzE5CTEwMTc0MjAwMwpjaHIxNQkxMDE3NDIwMjEJMTAxNzQzMjU1CmNocjE1CTEwMTc0MzM2MgkxMDE3NDM0MzQKY2hyMTUJMTAxNzQzNjk1CTEwMTc0NDEzNApjaHIxNQkxMDE3NDQyMzYJMTAxNzQ0NzYyCmNocjE1CTEwMTc0NDgwOAkxMDE3NDUwNjQKY2hyMTUJMTAxNzQ1MjMzCTEwMTc0NTUzNgpjaHIxNQkxMDE3NDU1NjAJMTAxNzQ1NzQzCmNocjE1CTEwMTc0NTc5MwkxMDE3NDYyNDQKY2hyMTYJNzQ5NzI1MzgJNzQ5NzI3NzAKY2hyMTYJNzQ5NzI4NzYJNzQ5NzM3NDMKY2hyMTYJNzQ5NzQwMjcJNzQ5NzgyNjkKY2hyMTYJNzQ5NzgzOTUJNzQ5NzkzMDQKY2hyMTYJNzQ5NzkzMjEJNzQ5Nzk0MTMKY2hyMTYJNzQ5Nzk1MjAJNzQ5Nzk3ODcKY2hyMTYJNzQ5Nzk5MTQJNzQ5ODE1NTYKY2hyMTYJNzQ5ODE3MTEJNzQ5ODU5MzQKY2hyMTYJNzQ5ODYwMzkJNzQ5ODcyMTEKY2hyMTYJNzQ5ODczMTMJNzQ5ODg2MDcKY2hyMTYJNzQ5ODg5ODEJNzQ5ODk2MTAKY2hyMTYJNzQ5ODk3ODMJNzQ5OTQyNDUKY2hyMTYJNzQ5OTQzNDgJNzQ5OTc1MzgKY2hyMTcJMzIxMDQwNzEJMzIxMDU2NTgKY2hyMTcJMzIxMDU3MDQJMzIxMDYxODQKY2hyMTcJMzIxMDYyMDIJMzIxMDY0ODEKY2hyMTcJMzIxMDY0OTgJMzIxMDc3NDgKY2hyMTcJMzIxMDc3NzEJMzIxMDc4NjQKY2hyMTcJMzIxMDc5NzQJMzIxMDg4MzgKY2hyMTcJMzIxMDg5NDAJMzIxMDkwMTAKY2hyMTcJMzIxMDkxMTMJMzIxMDk0NDUKY2hyMTcJMzIxMDk1NjIJMzIxMDk5NjEKY2hyMTcJMzIxMTAwNjQJMzIxMTA1MTIKY2hyMTcJMzIxMTA1MjkJMzIxMTMxMDEKY2hyMTcJMzIxMTMxNDMJMzIxMTM1NTgKY2hyMTcJMzIxMTM1NzcJMzIxMTM5MDIKY2hyMTcJMzIxMTQwNDMJMzIxMTQ0NDYKY2hyMTcJMzIxMTQ1NDkJMzIxMTQ2NjQKY2hyMTcJMzIxMTQ2OTMJMzIxMTQ5ODMKY2hyMTcJMzIxMTUwMDUJMzIxMTUyMzgKY2hyMTcJMzIxMTUyNzgJMzIxMTU1MDIKY2hyMTcJMzIxMTU2MDUJMzIxMTYzOTIKY2hyMTcJMzIxMTY0OTUJMzIxMTY3MjIKY2hyMTcJMzIxMTY3NDEJMzIxMTk0NjgKY2hyMTcJMzIxMTk1NzgJMzIxMjM0MTgKY2hyMTcJMzIxMjM4OTIJMzIxMjY5NzEKY2hyMTcJMzIxMjcwNzUJMzIxMjczMzIKY2hyMTcJMzIxMjczNTYJMzIxMjgwNjMKY2hyMTcJMzIxMjgxODkJMzIxMjgyNDUKY2hyMTcJMzIxMjgyNjQJMzIxMjgzNzQKY2hyMTcJMzIxMjg0MDEJMzIxMjg3NjMKY2hyMTcJMzIxMjg3ODEJMzIxMjkwMDEKY2hyMTcJMzIxMjkwMTgJMzIxMjkwNzEKY2hyMTcJODE5NDg1MzQJODE5NDk1NTIKY2hyMTcJODE5NDk4NDEJODE5NTAyMzYKY2hyMTcJODE5NTAzNTkJODE5NTEwNzMKY2hyMTcJODE5NTExNzUJODE5NTIwOTUKY2hyMTcJODE5NTI3MDMJODE5NTM3MjEKY2hyMTcJODE5NTM4MjQJODE5NTQzMDgKY2hyMTcJODE5NTQ0MTAJODE5NTU1ODYKY2hyMTcJODE5NTU5NzQJODE5NTg0NDAKY2hyMTcJODE5NTg3NzAJODE5NjE1MDAKY2hyMTcJODE5NjE2MDIJODE5Njc5ODEKY2hyMTcJODE5NjgwODMJODE5Njg0MTIKY2hyMTcJODE5Njg1MTQJODE5Njk5MjQKY2hyMTcJODE5NzA1NTYJODE5NzA2NjYKY2hyMTcJODE5NzA2OTYJODE5NzA4MDYKY2hyMTcJODE5NzA4MzEJODE5NzMwNzAKY2hyMTcJODE5NzMxNzIJODE5NzMzNzgKY2hyMTgJNDYwOTczMgk0NjA5ODIzCmNocjE4CTQ2MDk5MjUJNDYxNTg0NgpjaHIxOAk0NjE1ODY0CTQ2MTg4OTUKY2hyMTgJNDYxOTI5MAk0NjE5Nzg1CmNocjE4CTQ2MTk5MDMJNDYyMDIyNApjaHIxOAk0NjIwMzk5CTQ2MjEzMjYKY2hyMTgJNDYyMTQyOQk0NjM0NzMyCmNocjE4CTMwOTc1MzEzCTMwOTgxOTc0CmNocjE4CTMwOTgyMDgwCTMwOTg0NDYxCmNocjE4CTMwOTg0NjA5CTMwOTkyMTEwCmNocjE4CTMwOTkyMjI5CTMwOTk1OTIwCmNocjE4CTMwOTk2MTAzCTMxMDAwMzEzCmNocjE5CTI5NDE0NDgyCTI5NDIwNTY1CmNocjE5CTI5NDIxMTAzCTI5NDI1MDU3CmNocjE5CTI5NDI1MTY2CTI5NDI2NTk5CmNocjE5CTI5NDI2NzQwCTI5NDM4NDY2CmNocjE5CTI5NDM5NDUzCTI5NDM5NDgyCmNocjE5CTUxNjUwNjA2CTUxNjU1ODI2CmNocjE5CTUxNjU2MTc5CTUxNjU2NDE3CmNocjE5CTUxNjU2NzcwCTUxNjU3MTk1CmNocjE5CTUxNjU3MzA4CTUxNjYwNzQ1CmNocjE5CTUxNjYwOTI1CTUxNjYwOTg5CmNocjE5CTUxNjYxMTA4CTUxNjY2NTA0CmNocjE5CTUxNjY2ODEwCTUxNjcyMTMwCmNocjE5CTUxNjc0ODYxCTUxNjc0OTc3CmNocjE5CTUxNjc1MTAzCTUxNjc1NjA2CmNocjE5CTUzMjY5NzA0CTUzMjY5Nzc3CmNocjE5CTUzMjcwMzEzCTUzMjcwNjQ2CmNocjE5CTUzMjcwNzg4CTUzMjczMjA3CmNocjE5CTUzMjczMjU0CTUzMjc4MTY1CmNocjE5CTUzMjc4MjY3CTUzMjc4NTA5CmNocjE5CTUzMjc4NTI3CTUzMjc4NjcyCmNocjE5CTUzMjc4NjkxCTUzMjgwMzU2CmNocjE5CTUzMjgwNDU4CTUzMjgyNTk2CmNocjE5CTUzMjgyNjE4CTUzMjgyNzA3CmNocjE5CTUzMjgyODE0CTUzMjgzMDQzCmNocjE5CTUzMjgzMTQ1CTUzMjg1MzkwCmNocjE5CTUzMjg1NDkyCTUzMjg2ODg4CmNocjE5CTUzMjg3MDE3CTUzMjg3Mzc0CmNocjE5CTUzMjg3NTU1CTUzMjkzMTY4CmNocjE5CTUzMjkzMjcxCTUzMjk0NzA0CmNocjIyCTUwMzQxMDAwCTUwMzQyODkxCmNocjIyCTUwMzQ3MjU1CTUwMzQ4OTg2CmNocjIyCTUwMzQ5MDg4CTUwMzQ5MTQ0CmNocjIyCTUwMzQ5MjU1CTUwMzQ5MzEyCmNocjIyCTUwMzQ5NDE3CTUwMzUwNzkzCmNocjIyCTUwMzUxNTY2CTUwMzUxNjE3CmNocjIyCTUwMzUyMjQ1CTUwMzU0ODE3CmNocjIyCTUwMzU0OTIwCTUwMzU1MTc4CmNocjIyCTUwMzU1MzAwCTUwMzU1OTYyCmNocjIyCTUwMzU1OTg1CTUwMzU2MTMzCmNocjIyCTUwMzU2MTUwCTUwMzU2MjU0CmNocjIyCTUwMzU2MzU2CTUwMzU4OTM5CmNocjIyCTUwMzU5MDQxCTUwMzYwMjk5CmNocjIyCTUwMzYwNDAyCTUwMzY0MzE3Cg==', 'data/phase2/selected_intervals.tsv': '...', 'data/phase2/interval_panel.lock': 'cHJvamVjdD1Qcm9qZWN0IDAwMwpwaGFzZT0yQgppbnRlcnZhbF9zZWxlY3Rpb25fbWV0aG9kX3NoYTI1Nj1iM2RkOGZkYmQ4YzMwOGE5ZjkwOThmYmMyMTNhOWEyNDNjZWVjMDgxYjkyYjc0ODYyZDYwZDIxMmVhMTYxOWIyCmNvbnRleHRfYmVkc19kb3dubG9hZGVkX21hbmlmZXN0X3NoYTI1Nj05NzdiZjgxMTBkYjhlMjY0ODZlYWVhNTFkMTM4Y2QwMjI2NGRhNGIyNmIyOWZkOWMyOWM1OTVlM2JmNDA1Y2M1CnNoYXJlZF9jYWxsYWJsZV9zaGEyNTY9MjJhYmZmMTY0YzU3ZGMyNTMyN2Q1YTM5NWU1ZmMxN2Q0OWYzMGNiZGU0YjJkMThjYmM0NzQ4YTUyNjQzYzU0OQpzZWxlY3RlZF9pbnRlcnZhbHNfc2hhMjU2PTJlNWE2YTk3YzliYWNmNWZkZjNjZjU0N2FjN2RiMzJkMDZmNjExYTFmMmJjY2E2NmNiY2ZiZDA0ZTcyMmMwMWMKc2VsZWN0ZWRfYXNzZXNzYWJsZV9zZWdtZW50c19zaGEyNTY9NGM3N2UxN2MzZGVjNDcwNTFjYjliMjlmYTE0ZmVlNjdmMzFhNTU5Nzg1NDE3ZDA4M2Y0ZjdiY2FjYjA1Y2QyNgppbnRlcnZhbF9wYW5lbF9jaGVja3N1bXNfc2hhMjU2PTc3Mzg1NmViOTU4NDA1OTllM2VmMDgxZDJlZTNlMTVjMjU5MmNlMjNhMzQzYjdlZTY0YzA5MTVjODk5ODQzYTYKc2VsZWN0ZWRfd2luZG93cz01MApjb250ZXh0cz01CndpbmRvd3NfcGVyX2NvbnRleHQ9MTAKZXhjbHVkZWRfY2hyb21vc29tZT1jaHIyMApxdWVyeV92Y2ZzX2NvbnN1bHRlZD1GYWxzZQpiZW5jaG1hcmtfb3V0Y29tZXNfY29uc3VsdGVkPUZhbHNlCnN0YXR1cz1GUk9aRU5fQkVGT1JFX1FVRVJZX1ZDRl9ET1dOTE9BRF9BTkRfQkVOQ0hNQVJLSU5HCg==', 'docs/phase2/phase2_protocol.md': 'IyBQcm9qZWN0IDAwMyDigJQgUGhhc2UgMiBQcm90b2NvbAoKIyMgVGl0bGUKCk11bHRpLVNhbXBsZSwgTXVsdGktQ29udGV4dCBHZW5lcmFsaXphdGlvbiBvZiB0aGUgT21pY3NFZGdlIEJlbmNobWFyayBFdmlkZW5jZSBSZWdpc3RyeQoKIyMgU3RhdHVzCgpGUk9aRU4gQkVGT1JFIFBIQVNFIDIgU09VUkNFIFNFTEVDVElPTiBPUiBDT01QVVRBVElPTgoKIyMgUGhhc2UgMSBBdXRob3JpdGllcwoKUGhhc2UgMUEgYXV0aG9yaXRhdGl2ZSBhcmNoaXZlOgoKLSBwYXRoOiBgZGF0YS9waGFzZTFhL29taWNzZWRnZV9waGFzZTFhX3Jlc3VsdHMudGFyLmd6YAotIFNIQS0yNTY6IGA2NjMwODc5Nzc2YzNkYTIzMjZmNWM1NmI0ZGJiMTZjYTViZDQwOWI1Y2JjMDY1NTkwNjA4ZDQwYTk5YTI4MjAyYAoKUGhhc2UgMUIgYXV0aG9yaXRhdGl2ZSBhcmNoaXZlOgoKLSBwYXRoOiBgcmVzdWx0cy9waGFzZTFiL29taWNzZWRnZV9waGFzZTFiX3Jlc3VsdHMudGFyLmd6YAotIFNIQS0yNTY6IGA1MGJmNTk3Y2JjMTNhMDUxZDVkZGE3MjAxZTlkNzY1M2RiYzdiMTc2MmZlZjFiODMwMDE5OTY0MWRmNjQ3NGZiYAoKUGhhc2UgMUIgc2NpZW50aWZpYyBnYXRlOgoKLSAxNSAvIDE1IHZhbGlkYXRpb24gY3JpdGVyaWEgUEFTUwotIDEsNjY2IGJpb2xvZ2ljYWwgVkFSSUFOVCByZWNvcmRzCi0gMywzMzIgRVZFTlRfVkFSSUFOVF9MSU5LIHJlY29yZHMKLSBleGFjdCBub3JtYWxpemVkLWFsbGVsZSBpZGVudGl0eSBvbmx5Ci0gcmVwcmVzZW50YXRpb24tZXF1aXZhbGVuY2UgaW5mZXJlbmNlIGRpc2FibGVkCi0gbm8gcmVsaWFiaWxpdHkgb3IgY29uZmlkZW5jZSBzY29yZQoKLS0tCgojIDEuIFNjaWVudGlmaWMgUXVlc3Rpb24KCkNhbiB0aGUgT21pY3NFZGdlIEJlbmNobWFyayBFdmlkZW5jZSBSZWdpc3RyeSBnZW5lcmFsaXplIGJleW9uZCB0aGUgUGhhc2UgMSBIRzAwMiBjaHIyMCBwcm9vZi1vZi1jb25jZXB0IHdoaWxlIHByZXNlcnZpbmc6CgoxLiBpbW11dGFibGUgcnVuLXNjb3BlZCBiZW5jaG1hcmsgZXZpZGVuY2UsCjIuIGRldGVybWluaXN0aWMgYmlvbG9naWNhbCB2YXJpYW50IGlkZW50aXR5LAozLiBjb21wbGV0ZSBwcm92ZW5hbmNlLAo0LiBzYW1wbGUgYW5kIHRlY2hub2xvZ3kgY29udGV4dCwKNS4gZGlzYWdyZWVtZW50cyBiZXR3ZWVuIGJlbmNobWFyayBvYnNlcnZhdGlvbnMsCjYuIHJlcHJvZHVjaWJsZSBwcm9kdWN0IHF1ZXJpZXMsCjcuIGFic3RlbnRpb24gd2hlcmUgaWRlbnRpdHkgY2Fubm90IGJlIGVzdGFibGlzaGVkIHNhZmVseT8KClBoYXNlIDIgaXMgbm90IGEgYmVuY2htYXJrIGxlYWRlcmJvYXJkLgoKUGhhc2UgMiBpcyBub3QgaW50ZW5kZWQgdG8gZGV0ZXJtaW5lIHdoaWNoIHNlcXVlbmNpbmcgdGVjaG5vbG9neSBvciBjYWxsZXIgaXMgc3VwZXJpb3IuCgpQaGFzZSAyIHRlc3RzIHdoZXRoZXIgdGhlIGV2aWRlbmNlLXJlZ2lzdHJ5IGFyY2hpdGVjdHVyZSByZW1haW5zIHNjaWVudGlmaWNhbGx5IHZhbGlkIHdoZW4gdGhlIGFtb3VudCBhbmQgaGV0ZXJvZ2VuZWl0eSBvZiBwdWJsaWMgYmVuY2htYXJrIGV2aWRlbmNlIGluY3JlYXNlLgoKLS0tCgojIDIuIENvcmUgSHlwb3RoZXNpcwoKQSBiaW9sb2dpY2FsIHZhcmlhbnQgY2FuIGFjdCBhcyBhIHN0YWJsZSBhc3NlbWJseS1zcGVjaWZpYyBsb29rdXAgZW50aXR5IGFjcm9zcyBtdWx0aXBsZSBiZW5jaG1hcmsgcnVucyBhbmQgc2FtcGxlcyB3aGlsZSB0aGUgb3JpZ2luYWwgYmVuY2htYXJrIGV2ZW50cyBhbmQgb2JzZXJ2YXRpb25zIHJlbWFpbiBpbW11dGFibGUgYW5kIGluZGVwZW5kZW50bHkgaW50ZXJwcmV0YWJsZS4KClRoZSByZWdpc3RyeSBtdXN0IHBlcm1pdDoKClZBUklBTlQKICAtPiBFVkVOVF9WQVJJQU5UX0xJTksKICAgIC0+IEVWRU5UCiAgICAgIC0+IE9CU0VSVkFUSU9OCiAgICAgICAgLT4gQkVOQ0hNQVJLX1JVTgogICAgICAgICAgLT4gRVhQRVJJTUVOVAoKd2l0aG91dCBjb252ZXJ0aW5nIGhldGVyb2dlbmVvdXMgZXZpZGVuY2UgaW50byBhIGZvcmNlZCBjb25zZW5zdXMuCgotLS0KCiMgMy4gUGhhc2UgMiBBcmNoaXRlY3R1cmUKClBoYXNlIDIgTVVTVCByZXRhaW4gdGhlIGFyY2hpdGVjdHVyZSBlc3RhYmxpc2hlZCBpbiBQaGFzZXMgMUEgYW5kIDFCLgoKRXhpc3RpbmcgUGhhc2UgMUEgZW50aXRpZXMgcmVtYWluIGF1dGhvcml0YXRpdmU6CgotIEVYUEVSSU1FTlQKLSBCRU5DSE1BUktfUlVOCi0gRVZFTlQKLSBPQlNFUlZBVElPTgotIFNPVVJDRV9BUlRJRkFDVAotIFBST1ZFTkFOQ0VfTElOSwoKRXhpc3RpbmcgUGhhc2UgMUIgZW50aXRpZXMgcmVtYWluIGF1dGhvcml0YXRpdmU6CgotIFZBUklBTlQKLSBFVkVOVF9WQVJJQU5UX0xJTksKClBoYXNlIDIgbWF5IGFkZCBuZXcgcmVjb3JkcyB0byB0aGUgcmVnaXN0cnkuCgpQaGFzZSAyIE1VU1QgTk9UIG11dGF0ZSBleGlzdGluZyBQaGFzZSAxQSBvciBQaGFzZSAxQiBzY2llbnRpZmljIHJlY29yZHMuCgotLS0KCiMgNC4gQmlvbG9naWNhbCBWYXJpYW50IElkZW50aXR5CgpQaGFzZSAyIGluaGVyaXRzIHRoZSBmcm96ZW4gUGhhc2UgMUIgdjEgYmlvbG9naWNhbCBpZGVudGl0eSBkZWZpbml0aW9uLgoKQSBWQVJJQU5UIGlkZW50aXR5IGlzIGRldGVybWluZWQgYnk6CgotIGFzc2VtYmx5Ci0gY29udGlnCi0gbm9ybWFsaXplZCBzdGFydAotIG5vcm1hbGl6ZWQgZW5kCi0gbm9ybWFsaXplZCBSRUYKLSBub3JtYWxpemVkIEFMVAoKU2FtcGxlIGlkZW50aXR5IGlzIE5PVCBwYXJ0IG9mIHRoZSBWQVJJQU5UIGlkZW50aWZpZXIuCgpUZWNobm9sb2d5IGlzIE5PVCBwYXJ0IG9mIHRoZSBWQVJJQU5UIGlkZW50aWZpZXIuCgpDYWxsZXIgaXMgTk9UIHBhcnQgb2YgdGhlIFZBUklBTlQgaWRlbnRpZmllci4KCkJlbmNobWFyayBjb21wYXJhdG9yIGlzIE5PVCBwYXJ0IG9mIHRoZSBWQVJJQU5UIGlkZW50aWZpZXIuCgpUaGVzZSBhdHRyaWJ1dGVzIGJlbG9uZyB0byBldmlkZW5jZSBjb250ZXh0LCBub3QgYmlvbG9naWNhbCB2YXJpYW50IGlkZW50aXR5LgoKLS0tCgojIDUuIElkZW50aXR5IFN0YXRlcwoKQWxsb3dlZCBpZGVudGl0eSBzdGF0ZXMgcmVtYWluOgoKLSBFWEFDVF9OT1JNQUxJWkVEX0FMTEVMRQotIFJFUFJFU0VOVEFUSU9OX0VRVUlWQUxFTkNFX1NVUFBPUlRFRAotIE5PTl9FUVVJVkFMRU5UCi0gVU5SRVNPTFZFRAoKUGhhc2UgMiBNVVNUIE5PVCBlbWl0OgoKUkVQUkVTRU5UQVRJT05fRVFVSVZBTEVOQ0VfU1VQUE9SVEVECgp1bmxlc3MgYSBzZXBhcmF0ZSByZXByZXNlbnRhdGlvbi1lcXVpdmFsZW5jZSBhbGdvcml0aG0gaXM6CgoxLiBleHBsaWNpdGx5IHNwZWNpZmllZCwKMi4gdmFsaWRhdGVkIG9uIGFkdmVyc2FyaWFsIGNhc2VzLAozLiBmcm96ZW4gYmVmb3JlIHVzZSBvbiByZWFsIFBoYXNlIDIgZXZpZGVuY2UuCgpVbnRpbCB0aGVuLCByZXByZXNlbnRhdGlvbi1kaWZmZXJlbnQgY2FzZXMgcmVtYWluIFVOUkVTT0xWRUQuCgotLS0KCiMgNi4gUGhhc2UgMiBFeHBhbnNpb24gVGFyZ2V0CgpQaGFzZSAyIHNob3VsZCBtYXRlcmlhbGx5IGV4Y2VlZCB0aGUgUGhhc2UgMSBwcm9vZi1vZi1jb25jZXB0LgoKVGhlIGludGVuZGVkIHRhcmdldCBpczoKCi0gYXQgbGVhc3QgMyBkaXN0aW5jdCBiZW5jaG1hcmsgc2FtcGxlcyBpbiB0b3RhbCwKLSBhdCBsZWFzdCAyIHNlcXVlbmNpbmcgdGVjaG5vbG9naWVzIHdoZXJlIGNvbXBhdGlibGUgcHVibGljIGV2aWRlbmNlIGV4aXN0cywKLSBhdCBsZWFzdCAyIGJlbmNobWFyayBydW5zIHBlciBpbmNsdWRlZCBzYW1wbGUgd2hlcmUgcGFpcmVkIHRlY2hub2xvZ2llcyBhcmUgYXZhaWxhYmxlLAotIG11bHRpcGxlIGdlbm9taWMgY29udGV4dHMsCi0gbXVsdGlwbGUgY2hyb21vc29tZXMsCi0gYm90aCBTTlZzIGFuZCBpbmRlbHMsCi0gZGV0ZXJtaW5pc3RpYywgcHJvdmVuYW5jZS10cmFjZWFibGUgcmVnaW9uIHNlbGVjdGlvbi4KCkhHMDAyIG1heSByZW1haW4gaW5jbHVkZWQuCgpIRzAwMyBhbmQgSEcwMDQgYXJlIHByZWZlcnJlZCBhZGRpdGlvbmFsIHNhbXBsZXMgYmVjYXVzZSB0aGV5IGFyZSBlc3RhYmxpc2hlZCBHSUFCIGJlbmNobWFyayBnZW5vbWVzLCBidXQgdGhleSBhcmUgbm90IGF1dG9tYXRpY2FsbHkgYWNjZXB0ZWQuCgpUaGVpciBpbmNsdXNpb24gZGVwZW5kcyBvbiB0aGUgUGhhc2UgMiBzb3VyY2UgYXVkaXQuCgpJZiBIRzAwMyBvciBIRzAwNCBjYW5ub3Qgc2F0aXNmeSB0aGUgbG9ja2VkIHNvdXJjZSByZXF1aXJlbWVudHMsIGFub3RoZXIgcHVibGljIGJlbmNobWFyayBzYW1wbGUgbWF5IGJlIHN1YnN0aXR1dGVkIG9ubHkgdGhyb3VnaCBhIGRvY3VtZW50ZWQgUGhhc2UgMiBzb3VyY2Utc2VsZWN0aW9uIGRlY2lzaW9uLgoKTm8gc3Vic3RpdHV0aW9uIG1heSBiZSBiYXNlZCBvbiBiZW5jaG1hcmsgcGVyZm9ybWFuY2UuCgotLS0KCiMgNy4gU291cmNlIFJlcXVpcmVtZW50cwoKRXZlcnkgbmV3IFBoYXNlIDIgYmVuY2htYXJrIGV4cGVyaW1lbnQgTVVTVCBoYXZlIHN1ZmZpY2llbnQgcHVibGljIG1ldGFkYXRhIHRvIGlkZW50aWZ5LCB3aGVyZSBhcHBsaWNhYmxlOgoKLSBzYW1wbGUKLSBzZXF1ZW5jaW5nIHRlY2hub2xvZ3kKLSBwbGF0Zm9ybQotIHNvdXJjZSBjb3JwdXMKLSBzb3VyY2Ugc3VibWlzc2lvbgotIHJlZmVyZW5jZSBhc3NlbWJseQotIHZhcmlhbnQgY2FsbGVyIG9yIHN1Ym1pdHRlZCBjYWxsc2V0IHByb3ZlbmFuY2UKLSBiZW5jaG1hcmsgdHJ1dGggc291cmNlCi0gYmVuY2htYXJrIHJlZ2lvbnMKLSBjb21wYXJhdG9yCi0gY29tcGFyYXRvciB2ZXJzaW9uCi0gc291cmNlIGFydGlmYWN0IGNoZWNrc3VtCgpNaW5pbXVtIHJlcXVpcmVtZW50czoKCjEuIHB1YmxpYyBhY2Nlc3NpYmlsaXR5LAoyLiBzdGFibGUgc291cmNlIGlkZW50aXR5LAozLiBHUkNoMzggY29tcGF0aWJpbGl0eSwKNC4gdXNhYmxlIHZhcmlhbnQgY2FsbCBhcnRpZmFjdCwKNS4gdXNhYmxlIHRydXRoIGFydGlmYWN0LAo2LiB1c2FibGUgYmVuY2htYXJrLXJlZ2lvbiBkZWZpbml0aW9uLAo3LiByZXByb2R1Y2libGUgcHJvdmVuYW5jZS4KCkEgc291cmNlIGZhaWxpbmcgdGhlc2UgcmVxdWlyZW1lbnRzIE1VU1QgTk9UIHNpbGVudGx5IGVudGVyIHRoZSByZWdpc3RyeS4KCi0tLQoKIyA4LiBTb3VyY2UgU2VsZWN0aW9uIEluZGVwZW5kZW5jZQoKQmVuY2htYXJrIHBlcmZvcm1hbmNlIE1VU1QgTk9UIGJlIGNvbnN1bHRlZCB3aGVuIGRlY2lkaW5nIHdoaWNoIGNhbmRpZGF0ZSBkYXRhc2V0cyBlbnRlciBQaGFzZSAyLgoKU291cmNlIHNlbGVjdGlvbiBtYXkgdXNlOgoKLSBkYXRhIGF2YWlsYWJpbGl0eSwKLSBzYW1wbGUgaWRlbnRpdHksCi0gc2VxdWVuY2luZyB0ZWNobm9sb2d5LAotIHJlZmVyZW5jZSBhc3NlbWJseSwKLSBtZXRhZGF0YSBjb21wbGV0ZW5lc3MsCi0gZmlsZSBpbnRlZ3JpdHksCi0gdGVjaG5pY2FsIGNvbXBhdGliaWxpdHksCi0gbGljZW5zaW5nLAotIGNvbXB1dGUgZmVhc2liaWxpdHkuCgpTb3VyY2Ugc2VsZWN0aW9uIE1VU1QgTk9UIHVzZToKCi0gcHJlY2lzaW9uLAotIHJlY2FsbCwKLSBGMSwKLSBjb25jb3JkYW5jZSwKLSBmYWxzZS1wb3NpdGl2ZSBjb3VudCwKLSBmYWxzZS1uZWdhdGl2ZSBjb3VudCwKLSBhcHBhcmVudCBhZ3JlZW1lbnQgd2l0aCBhbm90aGVyIHRlY2hub2xvZ3ksCi0gdmlzdWFsbHkgYXR0cmFjdGl2ZSBiZW5jaG1hcmsgYmVoYXZpb3IuCgpUaGlzIHJ1bGUgZXhpc3RzIHRvIHByZXZlbnQgb3V0Y29tZS1kcml2ZW4gZGF0YXNldCBzZWxlY3Rpb24uCgotLS0KCiMgOS4gR2Vub21pYyBDb250ZXh0IEV4cGFuc2lvbgoKUGhhc2UgMiBtdXN0IGluY2x1ZGUgbXVsdGlwbGUgZ2Vub21pYyBjb250ZXh0cyByYXRoZXIgdGhhbiBzaW1wbHkgaW5jcmVhc2luZyB0aGUgc2l6ZSBvZiB0aGUgUGhhc2UgMSBpbnRlcnZhbC4KClRoZSBmaW5hbCBjb250ZXh0IHBhbmVsIG11c3QgaW5jbHVkZSwgd2hlcmUgdmFsaWQgcHVibGljIHN0cmF0aWZpY2F0aW9ucyBwZXJtaXQ6CgotIGNvbXBhcmF0aXZlbHkgdW5jb21wbGljYXRlZCBiZW5jaG1hcmsgc2VxdWVuY2UsCi0gaG9tb3BvbHltZXIgY29udGV4dCwKLSB0YW5kZW0tcmVwZWF0IGNvbnRleHQsCi0gbG93LW1hcHBhYmlsaXR5IG9yIGRpZmZpY3VsdC1tYXBwaW5nIGNvbnRleHQsCi0gYWRkaXRpb25hbCBub24tcmVwZXRpdGl2ZSBnZW5vbWljIGNvbnRleHQuCgpUaGUgc2FtZSBsb2NrZWQgY29udGV4dC1zZWxlY3Rpb24gcHJvY2VkdXJlIG11c3QgYmUgYXBwbGllZCBhY3Jvc3MgaW5jbHVkZWQgc2FtcGxlcyB3aGVyZXZlciBwb3NzaWJsZS4KCkNvbnRleHQgc2VsZWN0aW9uIE1VU1Qgb2NjdXIgYmVmb3JlIGJlbmNobWFyayBwZXJmb3JtYW5jZSBpcyBleGFtaW5lZC4KCi0tLQoKIyAxMC4gUmVnaW9uIFNlbGVjdGlvbgoKVGhlIGV4YWN0IFBoYXNlIDIgaW50ZXJ2YWxzIHdpbGwgYmUgZnJvemVuIG9ubHkgYWZ0ZXIgdGhlIHNvdXJjZSBhbmQgc3RyYXRpZmljYXRpb24gYXVkaXQuCgpUaGUgaW50ZXJ2YWwtc2VsZWN0aW9uIGFsZ29yaXRobSBNVVNUIGJlIGRldGVybWluaXN0aWMuCgpQZXJtaXR0ZWQgc2VsZWN0aW9uIGlucHV0cyBpbmNsdWRlOgoKLSBjaHJvbW9zb21lLAotIGNvb3JkaW5hdGUsCi0gcHVibGljIHN0cmF0aWZpY2F0aW9uIG1lbWJlcnNoaXAsCi0gY2FsbGFibGUgYmVuY2htYXJrLXJlZ2lvbiBpbnRlcnNlY3Rpb24sCi0gcmVxdWlyZWQgaW50ZXJ2YWwgc2l6ZSwKLSBkZXRlcm1pbmlzdGljIG9yZGVyaW5nIG9yIGhhc2hpbmcuCgpCZW5jaG1hcmsgb3V0Y29tZXMgTVVTVCBOT1QgYmUgc2VsZWN0aW9uIGlucHV0cy4KClRoZSBmaW5hbCBpbnRlcnZhbCBwYW5lbCBtdXN0IGJlIHdyaXR0ZW4gdG8gYSBmcm96ZW4gUGhhc2UgMiBpbnRlcnZhbCBtYW5pZmVzdCBiZWZvcmUgYmVuY2htYXJraW5nIGJlZ2lucy4KCi0tLQoKIyAxMS4gQmVuY2htYXJraW5nCgpQaGFzZSAyIHNob3VsZCByZXVzZSB0aGUgUGhhc2UgMSBjb21wYXJhdG9yIG1ldGhvZG9sb2d5IHdoZXJlIHRlY2huaWNhbGx5IGNvbXBhdGlibGUuCgpUaGUgY29tcGFyYXRvciBpbXBsZW1lbnRhdGlvbiBhbmQgdmVyc2lvbiBNVVNUIGJlIHJlY29yZGVkIGZvciBldmVyeSBCRU5DSE1BUktfUlVOLgoKQ2hhbmdpbmcgY29tcGFyYXRvciBzb2Z0d2FyZSwgbm9ybWFsaXphdGlvbiBsb2dpYywgdHJ1dGggaW50ZXJwcmV0YXRpb24sIG9yIGJlbmNobWFyay1yZWdpb24gc2VtYW50aWNzIHJlcXVpcmVzIGFuIGV4cGxpY2l0IHByb3RvY29sIGFtZW5kbWVudCBiZWZvcmUgdGhlIGFmZmVjdGVkIHJlYWwtZGF0YSBydW4uCgotLS0KCiMgMTIuIFNhbXBsZSBTZW1hbnRpY3MKCkEgYmlvbG9naWNhbCBWQVJJQU5UIG1heSBsZWdpdGltYXRlbHkgaGF2ZSBldmlkZW5jZSBmcm9tIG11bHRpcGxlIHNhbXBsZXMuCgpUaGVyZWZvcmU6CgpzYW1lIFZBUklBTlQKZG9lcyBOT1QgbWVhbgpzYW1lIGdlbm90eXBlCgphbmQ6CgpzYW1lIFZBUklBTlQKZG9lcyBOT1QgbWVhbgpzYW1lIGJlbmNobWFyayBvdXRjb21lCgphbmQ6CgpzYW1lIFZBUklBTlQKZG9lcyBOT1QgbWVhbgpzYW1lIHRlY2hub2xvZ3kgYmVoYXZpb3IuCgpFdmVyeSBwcm9kdWN0IHF1ZXJ5IG11c3QgcHJlc2VydmUgc2FtcGxlIGlkZW50aXR5IGV4cGxpY2l0bHkuCgotLS0KCiMgMTMuIFRlY2hub2xvZ3kgU2VtYW50aWNzCgpFdmlkZW5jZSBmcm9tIElsbHVtaW5hLCBPeGZvcmQgTmFub3BvcmUgVGVjaG5vbG9naWVzLCBvciBhbm90aGVyIGFkbWl0dGVkIHRlY2hub2xvZ3kgbXVzdCByZW1haW4gc2VwYXJhdGUgYXQgdGhlIG9ic2VydmF0aW9uL3J1biBsZXZlbC4KClBoYXNlIDIgTVVTVCBOT1QgZ2VuZXJhdGU6CgotIHRlY2hub2xvZ3kgd2lubmVyLAotIHRlY2hub2xvZ3kgcmFua2luZywKLSByZWxpYWJpbGl0eSBzY29yZSwKLSB0cnVzdCBzY29yZSwKLSBjb25maWRlbmNlIHNjb3JlLAotIGZvcmNlZCBjb25zZW5zdXMuCgpUaGUgcmVnaXN0cnkgc3RvcmVzIGVtcGlyaWNhbCBldmlkZW5jZS4KCkl0IGRvZXMgbm90IGRlY2lkZSB3aGljaCB0ZWNobm9sb2d5IHNob3VsZCBiZSB0cnVzdGVkLgoKLS0tCgojIDE0LiBQcm9kdWN0IFJlcXVpcmVtZW50CgpBIHVzZXIgbXVzdCBiZSBhYmxlIHRvIHF1ZXJ5IG9uZSBWQVJJQU5UIGFuZCByZXRyaWV2ZSBhbGwgbGlua2VkIGJlbmNobWFyayBldmlkZW5jZSBhdmFpbGFibGUgaW4gdGhlIHJlZ2lzdHJ5LCBpbmNsdWRpbmc6CgotIHNhbXBsZSwKLSB0ZWNobm9sb2d5LAotIGV4cGVyaW1lbnQsCi0gYmVuY2htYXJrIHJ1biwKLSBydW4tc2NvcGVkIGV2ZW50LAotIHRydXRoL3F1ZXJ5IHNpZGUsCi0gcmF3IGJlbmNobWFyayBkZWNpc2lvbiwKLSBjb21wYXJhdG9yL3ZlcnNpb24sCi0gc291cmNlIGFydGlmYWN0IHByb3ZlbmFuY2UsCi0gYmlvbG9naWNhbCBpZGVudGl0eSBtZXRob2QuCgpUaGUgcXVlcnkgbXVzdCBwcmVzZXJ2ZSBkaXNhZ3JlZW1lbnRzIHJhdGhlciB0aGFuIGNvbGxhcHNlIHRoZW0uCgotLS0KCiMgMTUuIFBoYXNlIDIgU3VicGhhc2VzCgojIyBQaGFzZSAyQSDigJQgU291cmNlIEF1ZGl0CgpJZGVudGlmeSBjYW5kaWRhdGUgcHVibGljIGRhdGFzZXRzIGFuZCBkZXRlcm1pbmUgd2hldGhlciB0aGV5IHNhdGlzZnkgdGhlIGZyb3plbiBzb3VyY2UgcmVxdWlyZW1lbnRzLgoKTm8gYmVuY2htYXJrIHBlcmZvcm1hbmNlIG1heSBiZSBjb25zdWx0ZWQgZHVyaW5nIHRoaXMgc3RhZ2UuCgpPdXRwdXQ6CgotIGNhbmRpZGF0ZSBzb3VyY2UgbWFuaWZlc3QKLSBlbGlnaWJpbGl0eSBkZWNpc2lvbgotIGV4Y2x1c2lvbiByZWFzb24KLSBpbW11dGFibGUgc291cmNlIGlkZW50aWZpZXJzCi0gY2hlY2tzdW1zIHdoZXJlIGF2YWlsYWJsZQoKIyMgUGhhc2UgMkIg4oCUIENvbnRleHQgUGFuZWwgRnJlZXplCgpDb25zdHJ1Y3QgdGhlIGRldGVybWluaXN0aWMgbXVsdGktY29udGV4dCBnZW5vbWljIGludGVydmFsIHBhbmVsLgoKT3V0cHV0OgoKLSBpbnRlcnZhbCBtYW5pZmVzdAotIHN0cmF0aWZpY2F0aW9uIHByb3ZlbmFuY2UKLSBzZWxlY3Rpb24gYWxnb3JpdGhtCi0gaW50ZXJ2YWwgY2hlY2tzdW1zCi0gZnJvemVuIHBhbmVsIGhhc2gKCiMjIFBoYXNlIDJDIOKAlCBCZW5jaG1hcmsgRXhwYW5zaW9uCgpSdW4gdGhlIGFwcHJvdmVkIGJlbmNobWFyayB3b3JrZmxvdyBvbmx5IG9uIGRhdGFzZXRzIGFuZCByZWdpb25zIGZyb3plbiBpbiAyQSBhbmQgMkIuCgpPdXRwdXQ6CgotIG5ldyBFWFBFUklNRU5UIHJlY29yZHMKLSBuZXcgQkVOQ0hNQVJLX1JVTiByZWNvcmRzCi0gbmV3IEVWRU5UIHJlY29yZHMKLSBuZXcgT0JTRVJWQVRJT04gcmVjb3JkcwotIHNvdXJjZS9wcm92ZW5hbmNlIHJlY29yZHMKCiMjIFBoYXNlIDJEIOKAlCBJZGVudGl0eSBFeHBhbnNpb24KCkFwcGx5IHRoZSBmcm96ZW4gYmlvbG9naWNhbCBpZGVudGl0eSBsYXllciB0byB0aGUgZXhwYW5kZWQgZXZpZGVuY2UuCgpPdXRwdXQ6CgotIG5ldyBWQVJJQU5UIHJlY29yZHMgd2hlcmUgbmVjZXNzYXJ5Ci0gbmV3IEVWRU5UX1ZBUklBTlRfTElOSyByZWNvcmRzCi0gdW5yZXNvbHZlZCBpZGVudGl0eSByZWNvcmRzIHdoZXJlIG5lY2Vzc2FyeQoKIyMgUGhhc2UgMkUg4oCUIFByb2R1Y3QgYW5kIEdlbmVyYWxpemF0aW9uIFZhbGlkYXRpb24KCkJ1aWxkIGFuZCB0ZXN0IGV4cGFuZGVkIHByb2R1Y3QgcXVlcmllcyBhbmQgZXZhbHVhdGUgdGhlIGZpbmFsIGZyb3plbiBQaGFzZSAyIGdhdGUuCgotLS0KCiMgMTYuIENvbXB1dGUgQ29uc3RyYWludAoKVGhlIHByb2plY3QgcmVtYWlucyBkZXNpZ25lZCBmb3IgbG93LWNvc3QgZXhlY3V0aW9uLgoKVGFyZ2V0IHBhaWQtY29tcHV0ZSBjb3N0OgoKQ0FEIDAgd2hlbmV2ZXIgcHJhY3RpY2FsLgoKTWF4aW11bSBhZGRpdGlvbmFsIHBhaWQgY29tcHV0ZSB3aXRob3V0IGFuIGV4cGxpY2l0IHByb3RvY29sIGFtZW5kbWVudDoKCkNBRCAxMDAuCgpQcmVmZXJyZWQgcmVzb3VyY2VzOgoKLSBsb2NhbCBtYWNPUyBmb3IgbWV0YWRhdGEsIHZhbGlkYXRpb24sIER1Y2tEQiwgc2NoZW1hcywgYW5kIHNtYWxsIHRyYW5zZm9ybWF0aW9ucywKLSBmcmVlIEdvb2dsZSBDb2xhYiB3aGVyZSBhZGRpdGlvbmFsIGNvbXB1dGUgaXMgcmVxdWlyZWQsCi0gcHVibGljbHkgaG9zdGVkIGFydGlmYWN0cywKLSBkZXRlcm1pbmlzdGljIHJlc3VtYWJsZSB3b3JrZmxvd3MuCgotLS0KCiMgMTcuIExvY2tlZCBQaGFzZSAyIFZhbGlkYXRpb24gQ3JpdGVyaWEKCjEuIFRoZSBhdXRob3JpdGF0aXZlIFBoYXNlIDFBIGFyY2hpdmUgcmVtYWlucyBieXRlLWlkZW50aWNhbCB0byBpdHMgZnJvemVuIFNIQS0yNTYuCgoyLiBUaGUgYXV0aG9yaXRhdGl2ZSBQaGFzZSAxQiBhcmNoaXZlIHJlbWFpbnMgYnl0ZS1pZGVudGljYWwgdG8gU0hBLTI1NiBgNTBiZjU5N2NiYzEzYTA1MWQ1ZGRhNzIwMWU5ZDc2NTNkYmM3YjE3NjJmZWYxYjgzMDAxOTk2NDFkZjY0NzRmYmAuCgozLiBObyBQaGFzZSAxQSBvciBQaGFzZSAxQiBzY2llbnRpZmljIHJlY29yZCBpcyBtb2RpZmllZCBieSBQaGFzZSAyLgoKNC4gRXZlcnkgYWRtaXR0ZWQgUGhhc2UgMiBzb3VyY2Ugc2F0aXNmaWVzIHRoZSBmcm96ZW4gc291cmNlIHJlcXVpcmVtZW50cyBhbmQgaGFzIGRvY3VtZW50ZWQgcHJvdmVuYW5jZS4KCjUuIERhdGFzZXQgYWRtaXNzaW9uIGlzIGluZGVwZW5kZW50IG9mIGJlbmNobWFyayBwZXJmb3JtYW5jZS4KCjYuIFRoZSBmaW5hbCBnZW5vbWljIGNvbnRleHQgcGFuZWwgaXMgZnJvemVuIGJlZm9yZSBQaGFzZSAyIGJlbmNobWFyayBvdXRjb21lcyBhcmUgZXhhbWluZWQuCgo3LiBSZWdpb24gc2VsZWN0aW9uIGlzIGRldGVybWluaXN0aWMgYW5kIHJlcHJvZHVjaWJsZSBmcm9tIHJlY29yZGVkIHB1YmxpYyBpbnB1dHMuCgo4LiBQaGFzZSAyIGNvbnRhaW5zIG11bHRpcGxlIGdlbm9taWMgY29udGV4dHMgYW5kIG11bHRpcGxlIGNocm9tb3NvbWVzLgoKOS4gUGhhc2UgMiBjb250YWlucyBib3RoIFNOViBhbmQgaW5kZWwgYmVuY2htYXJrIGV2aWRlbmNlLgoKMTAuIFRoZSBleHBhbmRlZCByZWdpc3RyeSBjb250YWlucyBhdCBsZWFzdCB0aHJlZSBkaXN0aW5jdCBiZW5jaG1hcmsgc2FtcGxlcyB1bmxlc3MgdGhlIHNvdXJjZSBhdWRpdCBwcm9kdWNlcyBhIGRvY3VtZW50ZWQgTU9ESUZZIHZlcmRpY3QgYmVmb3JlIGJlbmNobWFya2luZy4KCjExLiBXaGVyZSBwYWlyZWQgcHVibGljIGV2aWRlbmNlIHBlcm1pdHMsIGluY2x1ZGVkIHNhbXBsZXMgcHJlc2VydmUgZXZpZGVuY2UgZnJvbSBhdCBsZWFzdCB0d28gc2VxdWVuY2luZyB0ZWNobm9sb2dpZXMuCgoxMi4gQmlvbG9naWNhbCBWQVJJQU5UIGlkZW50aWZpZXJzIHJlbWFpbiBkZXRlcm1pbmlzdGljIGFuZCBhc3NlbWJseS1zcGVjaWZpYy4KCjEzLiBObyBFVkVOVCBpcyBzaWxlbnRseSBtZXJnZWQgYWNyb3NzIHJ1bnMsIHNhbXBsZXMsIHRlY2hub2xvZ2llcywgY2FsbGVycywgb3IgYmVuY2htYXJrIGV4ZWN1dGlvbnMuCgoxNC4gUmVwcmVzZW50YXRpb24tZGlmZmVyZW50IG9yIGFtYmlndW91cyBpZGVudGl0eSByZW1haW5zIFVOUkVTT0xWRUQgdW5sZXNzIGEgc2VwYXJhdGVseSBmcm96ZW4gYW5kIHZhbGlkYXRlZCBlcXVpdmFsZW5jZSBtZXRob2QgZXhpc3RzLgoKMTUuIEV2ZXJ5IG5ldyBFVkVOVF9WQVJJQU5UX0xJTksgaGFzIGNvbXBsZXRlIHByb3ZlbmFuY2UgdG8gaXRzIHNvdXJjZSBFVkVOVCwgQkVOQ0hNQVJLX1JVTiwgaWRlbnRpdHkgbWV0aG9kL3ZlcnNpb24sIGFuZCBvcmlnaW5hdGluZyBQaGFzZSAyIHNlbGVjdGlvbiByZWNvcmQuCgoxNi4gQSB2YXJpYW50LWxldmVsIHByb2R1Y3QgcXVlcnkgY2FuIHJldHVybiBldmlkZW5jZSBhY3Jvc3MgbXVsdGlwbGUgc2FtcGxlcyBhbmQgdGVjaG5vbG9naWVzIHdoaWxlIHByZXNlcnZpbmcgc2FtcGxlLCBleHBlcmltZW50LCBydW4sIGV2ZW50LCBvYnNlcnZhdGlvbiwgdHJ1dGgvcXVlcnkgc2lkZSwgcmF3IGRlY2lzaW9uLCBhbmQgcHJvdmVuYW5jZS4KCjE3LiBUaGUgZXhwYW5kZWQgcmVnaXN0cnkgaW50cm9kdWNlcyBubyByZWxpYWJpbGl0eSBzY29yZSwgY29uZmlkZW5jZSBzY29yZSwgdHJ1c3RlZCBsYWJlbCwgdGVjaG5vbG9neSByYW5raW5nLCBvciBmb3JjZWQgY29uc2Vuc3VzLgoKMTguIFJlLXJ1bm5pbmcgUGhhc2UgMiB0cmFuc2Zvcm1hdGlvbnMgb24gaWRlbnRpY2FsIGZyb3plbiBpbnB1dHMgcmVwcm9kdWNlcyBkZXRlcm1pbmlzdGljIGVudGl0eSBpZGVudGlmaWVycyBhbmQgc2NpZW50aWZpYyB0YWJsZSBjb250ZW50cy4KCjE5LiBBbGwgZmluYWwgUGhhc2UgMiByZWxlYXNlIGFydGlmYWN0cyBhcmUgY2hlY2tzdW0tbWFuaWZlc3RlZCBhbmQgaW5kZXBlbmRlbnRseSB2ZXJpZmlhYmxlIGFmdGVyIGFyY2hpdmUgZXh0cmFjdGlvbi4KCjIwLiBUaGUgZmluYWwgUGhhc2UgMiByZXBvcnQgc3RhdGVzIGxpbWl0YXRpb25zIGV4cGxpY2l0bHksIGluY2x1ZGluZyBkYXRhc2V0LXNlbGVjdGlvbiBzY29wZSwgYmVuY2htYXJrIHRydXRoIGxpbWl0YXRpb25zLCB1bnJlc29sdmVkIHJlcHJlc2VudGF0aW9uIGVxdWl2YWxlbmNlLCBhbmQgdGhlIGRpc3RpbmN0aW9uIGJldHdlZW4gZXZpZGVuY2UgYWdncmVnYXRpb24gYW5kIGJpb2xvZ2ljYWwgb3IgY2xpbmljYWwgdHJ1dGguCgotLS0KCiMgMTguIE91dGNvbWUgR2F0ZQoKUGhhc2UgMiByZWNlaXZlcyBvbmUgZmluYWwgdmVyZGljdDoKClBBU1MKCkFsbCBsb2NrZWQgcmVxdWlyZW1lbnRzIGFyZSBzYXRpc2ZpZWQgYW5kIHRoZSByZWdpc3RyeSBkZW1vbnN0cmF0ZXMgdmFsaWQgZ2VuZXJhbGl6YXRpb24gYmV5b25kIHRoZSBQaGFzZSAxIHByb29mLW9mLWNvbmNlcHQuCgpNT0RJRlkKClRoZSBhcmNoaXRlY3R1cmUgcmVtYWlucyBzY2llbnRpZmljYWxseSB2aWFibGUsIGJ1dCBvbmUgb3IgbW9yZSBub24tZmF0YWwgcmVxdWlyZW1lbnRzIHJlcXVpcmUgYSBkb2N1bWVudGVkIGNoYW5nZSBiZWZvcmUgcmVsZWFzZS4KClNUT1AKCkEgZnVuZGFtZW50YWwgYXNzdW1wdGlvbiBmYWlscywgaW5jbHVkaW5nIGJ1dCBub3QgbGltaXRlZCB0bzoKCi0gcHJvdmVuYW5jZSBjYW5ub3QgYmUgcHJlc2VydmVkLAotIGJlbmNobWFyayByZXN1bHRzIGFyZSByZXF1aXJlZCB0byBzZWxlY3QgZGF0YXNldHMgb3IgcmVnaW9ucywKLSBydW4tc2NvcGVkIGV2aWRlbmNlIGNhbm5vdCByZW1haW4gaW1tdXRhYmxlLAotIHZhcmlhbnQgaWRlbnRpdHkgcHJvZHVjZXMgdW5zYWZlIGNyb3NzLWFzc2VtYmx5IG9yIGFtYmlndW91cyBtZXJnZXMsCi0gcmVwcm9kdWNpYmlsaXR5IGNhbm5vdCBiZSBhY2hpZXZlZCwKLSBwcm9kdWN0IHF1ZXJpZXMgcmVxdWlyZSBzY2llbnRpZmljYWxseSB1bmp1c3RpZmllZCBldmlkZW5jZSBjb2xsYXBzZS4KCkNyaXRlcmlhIG1heSBub3QgYmUgd2Vha2VuZWQgYWZ0ZXIgcmVzdWx0cyBhcmUgb2JzZXJ2ZWQgc29sZWx5IHRvIG9idGFpbiBQQVNTLgoKLS0tCgojIDE5LiBOb24tR29hbHMKClBoYXNlIDIgZG9lcyBub3QgYXR0ZW1wdCB0bzoKCi0gYnVpbGQgYSBuZXcgdmFyaWFudCBjYWxsZXIsCi0gcmFuayBzZXF1ZW5jaW5nIHRlY2hub2xvZ2llcywKLSBpZGVudGlmeSBhIHVuaXZlcnNhbGx5IGJlc3QgY2FsbGVyLAotIGNyZWF0ZSBhIGNsaW5pY2FsIGludGVycHJldGF0aW9uIGRhdGFiYXNlLAotIGluZmVyIHBhdGhvZ2VuaWNpdHksCi0gaW5mZXIgcGF0aWVudC1sZXZlbCBjbGluaWNhbCB0cnV0aCwKLSBidWlsZCBhIHRydXN0IHNjb3JlLAotIGNsYWltIHRoYXQgYmVuY2htYXJrIHRydXRoIGlzIGFic29sdXRlIGJpb2xvZ2ljYWwgdHJ1dGgsCi0gc29sdmUgYXJiaXRyYXJ5IHJlcHJlc2VudGF0aW9uIGVxdWl2YWxlbmNlIHdpdGhvdXQgdmFsaWRhdGlvbiwKLSByZXBsYWNlIElHViBvciBvdGhlciBnZW5vbWUgYnJvd3NlcnMuCgotLS0KCiMgMjAuIEZyZWV6ZSBSdWxlCgpUaGlzIGRvY3VtZW50IG11c3QgYmUgaGFzaGVkIGJlZm9yZSBQaGFzZSAyIHNvdXJjZSBzZWxlY3Rpb24gYmVnaW5zLgoKQW55IGxhdGVyIHNjaWVudGlmaWMgY2hhbmdlIHJlcXVpcmVzOgoKMS4gYSBkb2N1bWVudGVkIGFtZW5kbWVudCwKMi4ganVzdGlmaWNhdGlvbiwKMy4gYSBuZXcgcHJvdG9jb2wgaGFzaCwKNC4gZXhwbGljaXQgaWRlbnRpZmljYXRpb24gb2Ygd2hldGhlciB0aGUgY2hhbmdlIG9jY3VycmVkIGJlZm9yZSBvciBhZnRlciBhZmZlY3RlZCByZXN1bHRzIHdlcmUgb2JzZXJ2ZWQuCg==', 'docs/phase2/phase2c_compute_protocol.md': 'IyBQcm9qZWN0IDAwMyDigJQgUGhhc2UgMkMgQ29tcHV0ZSBQcm90b2NvbAoKIyMgU3RhdHVzCgpGUk9aRU4gQkVGT1JFIFBIQVNFIDIgUVVFUlkgVkNGIFJFVFJJRVZBTCBBTkQgQkVOQ0hNQVJLIE9VVENPTUUgSU5TUEVDVElPTgoKIyMgUHVycG9zZQoKUGhhc2UgMkMgZXhlY3V0ZXMgdGhlIGZyb3plbiBQaGFzZSAyIGJlbmNobWFyayBwYW5lbCBmb3IgZm91ciBhZG1pdHRlZApleHBlcmltZW50IGNvbmRpdGlvbnM6CgotIEhHMDAzIElsbHVtaW5hCi0gSEcwMDMgT3hmb3JkIE5hbm9wb3JlCi0gSEcwMDQgSWxsdW1pbmEKLSBIRzAwNCBPeGZvcmQgTmFub3BvcmUKCkhlYXZ5IGdlbm9taWMgaW5wdXRzIGFuZCBjb21wYXJhdG9yIHdvcmtzcGFjZXMgYXJlIHRlbXBvcmFyeSBhbmQgcnVuIGluCkdvb2dsZSBDb2xhYi4KClRoZSBNYWMgYW5kIEdpdCByZXBvc2l0b3J5IHJldGFpbiBvbmx5IGNvbXBhY3QgcHJvdmVuYW5jZSwgbWFuaWZlc3RzLAp2YWxpZGF0ZWQgZXZpZGVuY2UsIGFuZCByZWxlYXNlIGFydGlmYWN0cy4KCi0tLQoKIyMgMS4gRnJvemVuIHVwc3RyZWFtIGF1dGhvcml0aWVzCgpQaGFzZSAyIHByb3RvY29sOgoKYGRvY3MvcGhhc2UyL3BoYXNlMl9wcm90b2NvbC5tZGAKClNIQS0yNTY6CgpgNjIzNDQ4NWU4OTE2ZmQzMGEzMzVjNzYxYTYyNjY4YjYxN2NiMTQ4YjViNWExNzUzZDEyZmQ3NzY1ZmExNjMzZmAKCkFkbWl0dGVkIFBoYXNlIDIgc291cmNlczoKCmBkYXRhL3BoYXNlMi9hZG1pdHRlZF9zb3VyY2VzLnRzdmAKClNIQS0yNTY6CgpgNTVlYmY2MzkwMWFjMGMxMThjYjVlYzY4N2I0OWNjODFjZjY4MGJiOTM1NzI0NmJhMmY2MTM4YjI3YTI5NjQxZmAKCkZyb3plbiBQaGFzZSAyIGludGVydmFsIHBhbmVsOgoKYGRhdGEvcGhhc2UyL3NlbGVjdGVkX2Fzc2Vzc2FibGVfc2VnbWVudHMuYmVkYAoKU0hBLTI1NjoKCmA0Yzc3ZTE3YzNkZWM0NzA1MWNiOWIyOWZhMTRmZWU2N2YzMWE1NTk3ODU0MTdkMDgzZjRmN2JjYWNiMDVjZDI2YAoKU2VsZWN0ZWQgaW50ZXJ2YWwgbWV0YWRhdGE6CgpgZGF0YS9waGFzZTIvc2VsZWN0ZWRfaW50ZXJ2YWxzLnRzdmAKClNIQS0yNTY6CgpgMmU1YTZhOTdjOWJhY2Y1ZmRmM2NmNTQ3YWM3ZGIzMmQwNmY2MTFhMWYyYmNjYTY2Y2JjZmJkMDRlNzIyYzAxY2AKCkludGVydmFsIHBhbmVsIGxvY2s6CgpgZGF0YS9waGFzZTIvaW50ZXJ2YWxfcGFuZWwubG9ja2AKClBhbmVsIGdlb21ldHJ5OgoKLSA1MCBub21pbmFsIHdpbmRvd3MKLSAxMCB3aW5kb3dzIHBlciBjb250ZXh0Ci0gNTQxIGFzc2Vzc2FibGUgQkVEIHNlZ21lbnRzCi0gMSwxNjIsNTcxIGFzc2Vzc2FibGUgYmFzZXMKLSAxOSByZXByZXNlbnRlZCBjaHJvbW9zb21lcwotIGNocjIwIGV4Y2x1ZGVkCgpObyBxdWVyeSBWQ0YgY29udGVudHMgb3IgYmVuY2htYXJrIG91dGNvbWVzIHdlcmUgY29uc3VsdGVkIGR1cmluZyBwYW5lbApjb25zdHJ1Y3Rpb24uCgotLS0KCiMjIDIuIEFkbWl0dGVkIHF1ZXJ5IFZDRnMKCkV4YWN0bHkgZm91ciBxdWVyeSBhcnRpZmFjdHMgYXJlIGFkbWl0dGVkLgoKIyMjIEhHMDAzIElsbHVtaW5hCgpTdWJtaXNzaW9uOgoKYDYwWjU5YAoKVVJMOgoKYGh0dHBzOi8vZGF0YS5uaXN0Lmdvdi9vZC9kcy9hcms6Lzg4NDM0L21kczItMjMzNi9zdWJtaXNzaW9uX3ZjZnMvNjBaNTkvNjBaNTlfSEcwMDMudmNmLmd6YAoKRXhwZWN0ZWQgU0hBLTI1NjoKCmBjNDM4NGI3MzRhZTdmNTRjZWVkYzNjNWU1YjUyZWY0YjQ4MDVjYmE4ZGNjNzAwNzlhNWRjZjQxMjAxMGFiNDRhYAoKIyMjIEhHMDAzIE94Zm9yZCBOYW5vcG9yZQoKU3VibWlzc2lvbjoKCmBSVTg4TmAKClVSTDoKCmBodHRwczovL2RhdGEubmlzdC5nb3Yvb2QvZHMvYXJrOi84ODQzNC9tZHMyLTIzMzYvc3VibWlzc2lvbl92Y2ZzL1JVODhOL1JVODhOX0hHMDAzLnZjZi5nemAKCkV4cGVjdGVkIFNIQS0yNTY6CgpgODRhYzVjNTI2YTkyNTQ4ZmE2ZjFjZTQ2MTA4Njc0YzI5ZDMyZWE3NTRmODYzYzllOTBmYmNhNDFhNDRlZGMyMGAKCiMjIyBIRzAwNCBJbGx1bWluYQoKU3VibWlzc2lvbjoKCmA2MFo1OWAKClVSTDoKCmBodHRwczovL2RhdGEubmlzdC5nb3Yvb2QvZHMvYXJrOi84ODQzNC9tZHMyLTIzMzYvc3VibWlzc2lvbl92Y2ZzLzYwWjU5LzYwWjU5X0hHMDA0LnZjZi5nemAKCkV4cGVjdGVkIFNIQS0yNTY6CgpgYjQxODRjMTY0ZWYyY2VmNjBiNTBmYTlkYjU3OWMyNTQ4ZWZhNWQ0MzAxZDEwNWE0OWY4ZDc4MDMxMTRlMWVmZmAKCiMjIyBIRzAwNCBPeGZvcmQgTmFub3BvcmUKClN1Ym1pc3Npb246CgpgUlU4OE5gCgpVUkw6CgpgaHR0cHM6Ly9kYXRhLm5pc3QuZ292L29kL2RzL2FyazovODg0MzQvbWRzMi0yMzM2L3N1Ym1pc3Npb25fdmNmcy9SVTg4Ti9SVTg4Tl9IRzAwNC52Y2YuZ3pgCgpFeHBlY3RlZCBTSEEtMjU2OgoKYGVjMGU0ODFkZGUwNDg2ZDMwYzlmODVjOTMyM2MxYmY3NTAwOGUyYmFkMmJhNTBlMmFjNDRlN2UxNTI3N2RhYmRgCgpFdmVyeSB3aG9sZSBxdWVyeSBhcnRpZmFjdCBtdXN0IGJlIGRvd25sb2FkZWQgYW5kIGl0cyBTSEEtMjU2IHZlcmlmaWVkCmJlZm9yZSBpdCBtYXkgYmUgaW5kZXhlZCwgc2xpY2VkLCBub3JtYWxpemVkLCBvciBiZW5jaG1hcmtlZC4KClRoZSBvcmlnaW5hbCBkb3dubG9hZGVkIGJ5dGVzIGFyZSBpbW11dGFibGUuCgotLS0KCiMjIDMuIEdJQUIgdHJ1dGgKClRydXRoIHJlbGVhc2U6CgpgR0lBQiB2NC4yLjFgCgpBc3NlbWJseToKCmBHUkNoMzhgCgojIyMgSEcwMDMgdHJ1dGggVkNGCgpgaHR0cHM6Ly9mdHAtdHJhY2UubmNiaS5ubG0ubmloLmdvdi9SZWZlcmVuY2VTYW1wbGVzL2dpYWIvcmVsZWFzZS9Bc2hrZW5hemltVHJpby9IRzAwM19OQTI0MTQ5X2ZhdGhlci9OSVNUdjQuMi4xL0dSQ2gzOC9IRzAwM19HUkNoMzhfMV8yMl92NC4yLjFfYmVuY2htYXJrLnZjZi5nemAKClRydXRoIGluZGV4OgoKc2FtZSBVUkwgcGx1cyBgLnRiaWAKCkJlbmNobWFyayBCRUQ6CgpgaHR0cHM6Ly9mdHAtdHJhY2UubmNiaS5ubG0ubmloLmdvdi9SZWZlcmVuY2VTYW1wbGVzL2dpYWIvcmVsZWFzZS9Bc2hrZW5hemltVHJpby9IRzAwM19OQTI0MTQ5X2ZhdGhlci9OSVNUdjQuMi4xL0dSQ2gzOC9IRzAwM19HUkNoMzhfMV8yMl92NC4yLjFfYmVuY2htYXJrX25vaW5jb25zaXN0ZW50LmJlZGAKCiMjIyBIRzAwNCB0cnV0aCBWQ0YKCmBodHRwczovL2Z0cC10cmFjZS5uY2JpLm5sbS5uaWguZ292L1JlZmVyZW5jZVNhbXBsZXMvZ2lhYi9yZWxlYXNlL0FzaGtlbmF6aW1UcmlvL0hHMDA0X05BMjQxNDNfbW90aGVyL05JU1R2NC4yLjEvR1JDaDM4L0hHMDA0X0dSQ2gzOF8xXzIyX3Y0LjIuMV9iZW5jaG1hcmsudmNmLmd6YAoKVHJ1dGggaW5kZXg6CgpzYW1lIFVSTCBwbHVzIGAudGJpYAoKQmVuY2htYXJrIEJFRDoKCmBodHRwczovL2Z0cC10cmFjZS5uY2JpLm5sbS5uaWguZ292L1JlZmVyZW5jZVNhbXBsZXMvZ2lhYi9yZWxlYXNlL0FzaGtlbmF6aW1UcmlvL0hHMDA0X05BMjQxNDNfbW90aGVyL05JU1R2NC4yLjEvR1JDaDM4L0hHMDA0X0dSQ2gzOF8xXzIyX3Y0LjIuMV9iZW5jaG1hcmtfbm9pbmNvbnNpc3RlbnQuYmVkYAoKRXZlcnkgcmV0cmlldmVkIHRydXRoIGFydGlmYWN0IHJlY2VpdmVzIGEgbG9jYWxseSBjb21wdXRlZCBTSEEtMjU2LgoKUHJldmlvdXNseSBmcm96ZW4gYmVuY2htYXJrLUJFRCBoYXNoZXMgbXVzdCByZXByb2R1Y2UgZXhhY3RseToKCkhHMDAzOgoKYDY1MmFmZDMwNDY3MDVhZjMyMDBmOWM4N2MyNTVmZWYxMWJiMjEyZGQ3NmM3NWExOTk5OWM5YjJkZjhhMzE4MGNgCgpIRzAwNDoKCmA4OGQxYzkyNmZkYzhhYmQ5YzM5YjNlOWZlMmFmM2ZjNDNkZDY3ZDI4NDY5MGEzZGM2MTI3YjM2OWQ5NmJjY2FkYAoKQSBtaXNtYXRjaCBpcyBhIGhhcmQgc3RvcC4KCi0tLQoKIyMgNC4gUmVmZXJlbmNlCgpSZWZlcmVuY2U6CgpgR0NBXzAwMDAwMTQwNS4xNV9HUkNoMzhfbm9fYWx0X2FuYWx5c2lzX3NldC5mYXN0YS5nemAKClVSTDoKCmBodHRwczovL2Z0cC10cmFjZS5uY2JpLm5sbS5uaWguZ292L1JlZmVyZW5jZVNhbXBsZXMvZ2lhYi9yZWxlYXNlL3JlZmVyZW5jZXMvR1JDaDM4L0dDQV8wMDAwMDE0MDUuMTVfR1JDaDM4X25vX2FsdF9hbmFseXNpc19zZXQuZmFzdGEuZ3pgCgpQdWJsaXNoZXIgTUQ1OgoKYDNhMzM0N2VhZTA4OTNmOTZlY2Y0OTVkMWMzOWUyMjg0YAoKRkFJIE1ENToKCmA1ZmRkYmMxMDljODI5ODBmOTQzNmFhNWMyMWE1N2M2MWAKCkdaSSBNRDU6CgpgY2ZhMWVlMTFiMWVjYjI5ZjkzNjU3OGIyNzAxNGZiZTBgCgpUaGUgcmVmZXJlbmNlIGlzIGRvd25sb2FkZWQgb25seSBpbnNpZGUgdGhlIGVwaGVtZXJhbCBDb2xhYiBydW50aW1lLgoKSXRzIE1ENSBhbmQgbG9jYWxseSBjb21wdXRlZCBTSEEtMjU2IG11c3QgYmUgcmVjb3JkZWQgYmVmb3JlIGNvbXBhcmlzb24uCgotLS0KCiMjIDUuIENvbXBhcmF0b3IgaWRlbnRpdHkKClNvZnR3YXJlIGlkZW50aXR5IHJlbWFpbnMgaW5oZXJpdGVkIGZyb206CgpgZW52aXJvbm1lbnQvY29tcGFyYXRvci1sb2NrLnlhbWxgCgpoYXAucHk6CgpgMC4zLjE1YAoKQmlvQ29udGFpbmVyIG1hbmlmZXN0OgoKYHNoYTI1NjpkNjNiOTYzYTZjYjAxYjQ4MzAzOTNiMjIzNjllN2I5MWQyOThlNDE1NmRkZTM1MzczOWU3NGU0Y2ZhNGY5NmQwYAoKUlRHIFRvb2xzIC8gdmNmZXZhbDoKCmAzLjEyLjFgCgpSVEcgYXJjaGl2ZSBTSEEtMjU2OgoKYGJhNDNjYWRiYjNlNzliZmY3NGU3ZDJmODU0MGRkNWUyMDE1NzRlYzhlMjkzYmE1NTVmM2FlZWZkMTM3NWRmYzVgCgpDb21wYXJpc29uIGVuZ2luZToKCmB2Y2ZldmFsYAoKQW5ub3RhdGVkIGNvbXBhcmF0b3Igb3V0cHV0OgoKZW5hYmxlZCB3aXRoIGAtVmAKClRocmVhZHM6CgpgMmAKCi0tLQoKIyMgNi4gRXhwbGljaXQgUGhhc2UgMSDihpIgUGhhc2UgMiBjb21wYXJhdG9yIGFkYXB0YXRpb24KClRoZSBmb2xsb3dpbmcgZmllbGRzIGluIGBlbnZpcm9ubWVudC9jb21wYXJhdG9yLWxvY2sueWFtbGAgYXJlClBoYXNlLTEtc3BlY2lmaWMgYW5kIE1VU1QgTk9UIGJlIHVzZWQgZm9yIFBoYXNlIDJDOgoKLSBgY29tcGFyaXNvbi5jb3JlX2ludGVydmFsYAotIGBjb21wYXJpc29uLnBhZGRlZF9pbnRlcnZhbGAKClRob3NlIGZpZWxkcyBkZXNjcmliZSB0aGUgaGlzdG9yaWNhbCBIRzAwMiBjaHIyMCBQaGFzZSAxQSBleHBlcmltZW50LgoKUGhhc2UgMkMgcmVwbGFjZXMgdGhvc2UgY29vcmRpbmF0ZXMgd2l0aDoKCmBkYXRhL3BoYXNlMi9zZWxlY3RlZF9hc3Nlc3NhYmxlX3NlZ21lbnRzLmJlZGAKCk5vIFBoYXNlIDEgY2hyMjAgaW50ZXJ2YWwgaXMgcGVybWl0dGVkIGluIFBoYXNlIDJDIGV4ZWN1dGlvbi4KCi0tLQoKIyMgNy4gUmV0cmlldmFsIHBhZGRpbmcKCklucHV0IHJldHJpZXZhbCB1c2VzOgoKYDEsMDAwIGJwYAoKb2YgZ2Vub21pYyBwYWRkaW5nIGFyb3VuZCBlYWNoIGFzc2Vzc2FibGUgc2VnbWVudC4KClBhZGRpbmcgZG9lcyBub3QgZW5sYXJnZSB0aGUgYmVuY2htYXJrIHBhbmVsIG9yIGV4cG9ydGVkIGV2aWRlbmNlLgoKVGhlIGV4YWN0IGZyb3plbiBwYW5lbCByZW1haW5zIHRoZSBoYXAucHkgYC1mYCBiZW5jaG1hcmstcmVnaW9uIGlucHV0LgoKQm91bmRhcnktb25seSByZWNvcmRzIG1heSBiZSByZXRhaW5lZCBhcyBkaWFnbm9zdGljcyBidXQgbXVzdCBub3QgYmVjb21lCmNvcmUgcmVnaXN0cnkgZXZpZGVuY2UuCgotLS0KCiMjIDguIFF1ZXJ5IHByZXBhcmF0aW9uCgpGb3IgZWFjaCBxdWVyeSBWQ0Y6CgoxLiBkb3dubG9hZCB0aGUgd2hvbGUgYXJ0aWZhY3QsCjIuIHZlcmlmeSBmcm96ZW4gU0hBLTI1NiwKMy4gcmV0YWluIG9yaWdpbmFsIGJ5dGVzIHVuY2hhbmdlZCwKNC4gdmFsaWRhdGUgQkdaRi9WQ0YgcmVhZGFiaWxpdHksCjUuIGNyZWF0ZSBkZXJpdmVkIGluZGV4IGlmIHBvc3NpYmxlLAo2LiBkZXJpdmUgYSBwYWRkZWQtcGFuZWwgcXVlcnkgc2xpY2UsCjcuIHJlY29yZCBjaGVja3N1bXMgYW5kIHRyYW5zZm9ybWF0aW9uIGNvbW1hbmRzLgoKT3JpZ2luYWwgcXVlcnkgZmlsZXMgbXVzdCBuZXZlciBiZSBvdmVyd3JpdHRlbi4KCi0tLQoKIyMgOS4gVHJ1dGggcHJlcGFyYXRpb24KCkZvciBIRzAwMyBhbmQgSEcwMDQ6CgoxLiByZXRyaWV2ZSB0cnV0aCBWQ0YsCjIuIHJldHJpZXZlIFRCSSwKMy4gcmV0cmlldmUgYmVuY2htYXJrIEJFRCwKNC4gdmVyaWZ5IGZyb3plbiBiZW5jaG1hcmstQkVEIFNIQS0yNTYsCjUuIGNvbXB1dGUgU0hBLTI1NiBmb3IgdHJ1dGggVkNGIGFuZCBUQkksCjYuIGRlcml2ZSBwYWRkZWQtcGFuZWwgdHJ1dGggc2xpY2UsCjcuIHJldGFpbiB0cmFuc2Zvcm1hdGlvbiBjb21tYW5kcyBhbmQgaGFzaGVzLgoKVGhlIHNhbWUgdHJ1dGggc291cmNlIGlzIHNoYXJlZCBieSBJbGx1bWluYSBhbmQgT05UIGZvciBlYWNoIHNhbXBsZS4KCi0tLQoKIyMgMTAuIFJlZmVyZW5jZSBwcmVwYXJhdGlvbgoKVGhlIGV4YWN0IEdSQ2gzOCBuby1hbHQgcmVmZXJlbmNlIGlzIHVzZWQuCgpUb29sLXNwZWNpZmljIHN0cnVjdHVyZXMgaW5jbHVkaW5nIGFuIFJURyBTREYgYXJlIGRlcml2ZWQgYXJ0aWZhY3RzLgoKSGFzaGVzIGFuZCBkZXJpdmF0aW9uIGNvbW1hbmRzIG11c3QgYmUgY2FwdHVyZWQuCgpSZWZlcmVuY2Ugc2VxdWVuY2UgaWRlbnRpdHkgbXVzdCBub3QgY2hhbmdlLgoKLS0tCgojIyAxMS4gQmVuY2htYXJrIHJ1bnMKCkV4YWN0bHkgZm91ciBQaGFzZSAyQyBjb21wYXJhdG9yIHJ1bnMgYXJlIGV4ZWN1dGVkOgoKLSBIRzAwMyAvIDYwWjU5IC8gSWxsdW1pbmEKLSBIRzAwMyAvIFJVODhOIC8gT05UCi0gSEcwMDQgLyA2MFo1OSAvIElsbHVtaW5hCi0gSEcwMDQgLyBSVTg4TiAvIE9OVAoKRWFjaCBydW4gdXNlcyB0aGUgY29ycmVzcG9uZGluZyBxdWVyeSwgY29ycmVzcG9uZGluZyBzYW1wbGUgdHJ1dGgsCnNhbWUgR1JDaDM4IHJlZmVyZW5jZSwgc2FtZSBmcm96ZW4gcGFuZWwsIGhhcC5weSAwLjMuMTUsClJURyB2Y2ZldmFsIDMuMTIuMSwgYW5kIHR3byB0aHJlYWRzLgoKTm8gcXVhbGl0eSB0aHJlc2hvbGQsIGNhbGxlci1zcGVjaWZpYyB0aHJlc2hvbGQsIHRlY2hub2xvZ3ktc3BlY2lmaWMgZmlsdGVyLApvciBwb3N0LWhvYyByZWdpb24gcmVwbGFjZW1lbnQgbWF5IGJlIGludHJvZHVjZWQuCgotLS0KCiMjIDEyLiBDb21wYXJhdG9yIGNvbW1hbmQgc2VtYW50aWNzCgpTZW1hbnRpYyBjb21tYW5kIGNvbnRyYWN0OgoKICAgIGhhcC5weSBUUlVUSF9TTElDRSBRVUVSWV9TTElDRQogICAgICAtZiBGUk9aRU5fUEhBU0UyX1BBTkVMX0JFRAogICAgICAtciBHUkNIMzhfUkVGRVJFTkNFCiAgICAgIC1vIFJVTl9PVVRQVVRfUFJFRklYCiAgICAgIC1WCiAgICAgIC0tZW5naW5lPXZjZmV2YWwKICAgICAgLS1lbmdpbmUtdmNmZXZhbC1wYXRoPS9vcHQvcnRnLXRvb2xzLTMuMTIuMS9ydGcKICAgICAgLS1lbmdpbmUtdmNmZXZhbC10ZW1wbGF0ZSBHUkNIMzhfUlRHX1NERgogICAgICAtLXRocmVhZHM9MgoKUGF0aCBuYW1lcyBhcmUgcnVudGltZS1sb2NhbC4KClNjaWVudGlmaWMgYXJndW1lbnRzIGFyZSBmcm96ZW4uCgotLS0KCiMjIDEzLiBSdW50aW1lCgpQaGFzZSAyQyBydW5zIGluIGEgZnJlc2ggR29vZ2xlIENvbGFiIENQVSBydW50aW1lLgoKTWluaW11bSBwcmVmbGlnaHQ6CgotIExpbnV4IGFtZDY0IC8geDg2XzY0Ci0gPj0gMiBsb2dpY2FsIENQVXMKLSA+PSA3IEdpQiBSQU0KLSA+PSAxNSBHaUIgZnJlZSB0ZW1wb3JhcnkgZGlzawoKTGFyZ2UgZ2Vub21pYyBmaWxlcyByZW1haW4gaW4gZXBoZW1lcmFsIGAvY29udGVudGAuCgpUaGUgcGlubmVkIHJvb3RsZXNzIHJ1bm5lciBmcm9tIHRoZSBwcmlvciBwcm9vZiByZW1haW5zOgoKYHVkb2NrZXI9PTEuMy4xN2AKCi0tLQoKIyMgMTQuIFJ1bnRpbWUgaWRlbnRpdHkgY2FwdHVyZQoKQ2FwdHVyZToKCi0gYXJjaGl0ZWN0dXJlCi0gQ1BVIGNvdW50Ci0gUkFNCi0gZnJlZSBkaXNrCi0gUHl0aG9uCi0gaGFwLnB5IGlkZW50aXR5IGV2aWRlbmNlCi0gY29udGFpbmVyIGRpZ2VzdAotIFJURwotIEphdmEKLSBiY2Z0b29scwotIHNhbXRvb2xzCi0gdWRvY2tlcgotIGNvbW1hbmQgYXJndgotIHRpbWVzdGFtcHMKLSBzdGRvdXQKLSBzdGRlcnIKLSBleGl0IHN0YXR1cwotIHdhbGwgdGltZQotIG91dHB1dCBoYXNoZXMKCi0tLQoKIyMgMTUuIE91dGNvbWUgaXNvbGF0aW9uCgpCZWZvcmUgY29tcGFyaXNvbiwgb25seSBzdHJ1Y3R1cmFsL3Byb3ZlbmFuY2UgaW5mb3JtYXRpb24gbWF5IGJlIGluc3BlY3RlZC4KCkRvIG5vdCBpbnNwZWN0IG9yIHN1bW1hcml6ZToKCi0gVFAKLSBGUAotIEZOCi0gcHJlY2lzaW9uCi0gcmVjYWxsCi0gRjEKLSB0ZWNobm9sb2d5IGFncmVlbWVudAotIG91dGNvbWUgZGlzdHJpYnV0aW9ucwoKdW50aWwgYWxsIGZvdXIgc291cmNlcyBhbmQgY29tcGFyYXRvciBjb25maWd1cmF0aW9uIHBhc3MgcHJlZmxpZ2h0LgoKTm8gc291cmNlIG9yIHJlZ2lvbiBtYXkgYmUgcmVwbGFjZWQgYmVjYXVzZSBvZiBwZXJmb3JtYW5jZS4KCi0tLQoKIyMgMTYuIEV2aWRlbmNlIGV4cG9ydAoKUGhhc2UgMkMgcHJlc2VydmVzOgoKLSBleHBlcmltZW50Ci0gYmVuY2htYXJrIHJ1bgotIGV2ZW50Ci0gdHJ1dGggb2JzZXJ2YXRpb24KLSBxdWVyeSBvYnNlcnZhdGlvbgotIHNvdXJjZSBhcnRpZmFjdHMKLSBwcm92ZW5hbmNlIGxpbmtzCgpFdmVudHMgcmVtYWluIHJ1bi1zY29wZWQuCgpDcm9zcy1ydW4gYmlvbG9naWNhbCBpZGVudGl0eSBpcyBkZWZlcnJlZCB0byBQaGFzZSAyRC4KCi0tLQoKIyMgMTcuIFJlcXVpcmVkIGNvbXBhY3QgQ29sYWIgb3V0cHV0cwoKUmV0dXJuIG9ubHkgY29tcGFjdCByZXByb2R1Y2liaWxpdHkgYW5kIGV2aWRlbmNlIGFydGlmYWN0czoKCi0gcnVudGltZSBpZGVudGl0eQotIHJldHJpZXZhbCBtYW5pZmVzdAotIHNvdXJjZS9kZXJpdmVkIGNoZWNrc3VtcwotIGNvbW1hbmQgbG9nCi0gcnVuIG1hbmlmZXN0Ci0gYmVuY2htYXJrIHJ1bnMKLSBleHBlcmltZW50cwotIHJlcXVpcmVkIGFubm90YXRlZCBjb21wYXJhdG9yIG91dHB1dAotIGV2ZW50cwotIG9ic2VydmF0aW9ucwotIHByb3ZlbmFuY2UgbGlua3MKLSB2YWxpZGF0aW9uIHJlcG9ydAotIHN0ZG91dC9zdGRlcnIKLSBwYWNrYWdlIGNoZWNrc3VtIG1hbmlmZXN0CgpEbyBub3QgcmV0dXJuIGxhcmdlIHF1ZXJ5IFZDRnMsIHRydXRoIFZDRnMsIHJlZmVyZW5jZSBGQVNUQSwKY29udGFpbmVyIGxheWVycywgUlRHIFNERiwgb3IgdGVtcG9yYXJ5IHdvcmtzcGFjZXMuCgotLS0KCiMjIDE4LiBGYWlsLWNsb3NlZCBjb25kaXRpb25zCgpTdG9wIGJlZm9yZSBzY2llbnRpZmljIGludGVycHJldGF0aW9uIG9uOgoKLSBxdWVyeSBTSEEtMjU2IG1pc21hdGNoCi0gYmVuY2htYXJrIEJFRCBTSEEtMjU2IG1pc21hdGNoCi0gcmVmZXJlbmNlIE1ENSBtaXNtYXRjaAotIGNvbXBhcmF0b3IgaWRlbnRpdHkgbWlzbWF0Y2gKLSBSVEcgYXJjaGl2ZSBTSEEtMjU2IG1pc21hdGNoCi0gZnJvemVuIHBhbmVsIFNIQS0yNTYgbWlzbWF0Y2gKLSBtaXNzaW5nIHJ1bgotIGNvbXBhcmF0b3Igbm9uLXplcm8gZXhpdAotIGluY29tcGxldGUgcHJvdmVuYW5jZQotIG91dGNvbWUtZHJpdmVuIHNvdXJjZSByZXBsYWNlbWVudAotIGFjY2lkZW50YWwgUGhhc2UgMSBjaHIyMCB1c2FnZQoKLS0tCgojIyAxOS4gQ29tcHV0ZS9zdG9yYWdlIHBvbGljeQoKTGFyZ2UgZmlsZXMgZXhpc3Qgb25seSBpbiBDb2xhYi4KClRhcmdldCBwYWlkIGNvbXB1dGU6CgpgQ0FEICQwYAoKTm8gbGFyZ2UgcmF3IFBoYXNlIDIgZ2Vub21pYyBpbnB1dCBpcyByZXF1aXJlZCBvbiB0aGUgZGV2ZWxvcG1lbnQgTWFjLgoKLS0tCgojIyAyMC4gRnJlZXplIHJ1bGUKClRoaXMgcHJvdG9jb2wgbXVzdCBiZSBoYXNoZWQgYW5kIGNvbW1pdHRlZCBiZWZvcmU6CgotIFBoYXNlIDIgcXVlcnkgVkNGIHJldHJpZXZhbAotIFBoYXNlIDIgYmVuY2htYXJrIGV4ZWN1dGlvbgotIFBoYXNlIDIgb3V0Y29tZSBpbnNwZWN0aW9uCgpBbnkgc2NpZW50aWZpYyBjaGFuZ2UgYWZ0ZXIgb3V0Y29tZSBleHBvc3VyZSByZXF1aXJlcyBhIGRvY3VtZW50ZWQgYW1lbmRtZW50CmFuZCBhIG5ldyBoYXNoLgo=', 'docs/phase2/phase2c_compute_protocol.lock': 'cHJvamVjdD1Qcm9qZWN0IDAwMwpwaGFzZT0yQwpwcm90b2NvbD1kb2NzL3BoYXNlMi9waGFzZTJjX2NvbXB1dGVfcHJvdG9jb2wubWQKcHJvdG9jb2xfc2hhMjU2PWUyZWIyYmI5MmM4OWIxNDk3YTEwZDBmMGI2ZDU3YjkxYjk1YzIwYjc5MTkzZTllMGZkMjU1NDAzMDI3OTYxYmUKcGhhc2UyX3Byb3RvY29sX3NoYTI1Nj02MjM0NDg1ZTg5MTZmZDMwYTMzNWM3NjFhNjI2NjhiNjE3Y2IxNDhiNWI1YTE3NTNkMTJmZDc3NjVmYTE2MzNmCmFkbWl0dGVkX3NvdXJjZXNfc2hhMjU2PTU1ZWJmNjM5MDFhYzBjMTE4Y2I1ZWM2ODdiNDljYzgxY2Y2ODBiYjkzNTcyNDZiYTJmNjEzOGIyN2EyOTY0MWYKc2VsZWN0ZWRfaW50ZXJ2YWxzX3NoYTI1Nj0yZTVhNmE5N2M5YmFjZjVmZGYzY2Y1NDdhYzdkYjMyZDA2ZjYxMWExZjJiY2NhNjZjYmNmYmQwNGU3MjJjMDFjCnNlbGVjdGVkX2Fzc2Vzc2FibGVfc2VnbWVudHNfc2hhMjU2PTRjNzdlMTdjM2RlYzQ3MDUxY2I5YjI5ZmExNGZlZTY3ZjMxYTU1OTc4NTQxN2QwODNmNGY3YmNhY2IwNWNkMjYKaGFwX3B5X3ZlcnNpb249MC4zLjE1CnJ0Z192Y2ZldmFsX3ZlcnNpb249My4xMi4xCnJldHJpZXZhbF9wYWRkaW5nX2JwPTEwMDAKcnVuX2NvdW50PTQKcXVlcnlfdmNmc19jb25zdWx0ZWQ9RmFsc2UKYmVuY2htYXJrX291dGNvbWVzX2NvbnN1bHRlZD1GYWxzZQpwaGFzZTFfY2hyMjBfaW50ZXJ2YWxfYWxsb3dlZD1GYWxzZQpsYXJnZV9jb21wdXRlX2xvY2F0aW9uPUdPT0dMRV9DT0xBQl9FUEhFTUVSQUwKc3RhdHVzPUZST1pFTl9CRUZPUkVfUEhBU0UyQ19SRVRSSUVWQUxfQU5EX0NPTVBVVEUK', 'docs/phase2/benchmark_run_schema_extension.lock': 'cHJvamVjdD1Qcm9qZWN0IDAwMwpwaGFzZT0yQwpzY2hlbWFfbmFtZT1CRU5DSE1BUktfUlVOCnNjaGVtYV92ZXJzaW9uPTIuMC4wCnNjaGVtYV9wYXRoPXNjaGVtYXMvcGhhc2UyL2JlbmNobWFya19ydW4uc2NoZW1hLmpzb24Kc2NoZW1hX3NoYTI1Nj0yZGIzMzEwYThmNTJkYzMwZjA2NjhiNWNmMGJkMjVmYjkxYjFiZWQzNWY4YzExMTExOTFiODBkMzg0MTBjMmQ3CnJhdGlvbmFsZV9wYXRoPWRvY3MvcGhhc2UyL2JlbmNobWFya19ydW5fc2NoZW1hX2V4dGVuc2lvbi5tZApyYXRpb25hbGVfc2hhMjU2PTA2ODk5MjczZjYxNjhjMWRkMGQ0OGM0MzJlMDg3ZjA2NjBmNjU2ZWFhZmMwZTNiOTE3Y2U4YmNiY2ZiYmFjMDkKcGhhc2UxX3NjaGVtYV9tb2RpZmllZD1GYWxzZQpiZW5jaG1hcmtfcmVnaW9uX3NldF9zaGEyNTY9NGM3N2UxN2MzZGVjNDcwNTFjYjliMjlmYTE0ZmVlNjdmMzFhNTU5Nzg1NDE3ZDA4M2Y0ZjdiY2FjYjA1Y2QyNgpzZWxlY3RlZF93aW5kb3dzPTUwCnNlZ21lbnRfY291bnQ9NTQxCmNvbnRleHRfY291bnQ9NQpjaHJvbW9zb21lX2NvdW50PTE5CmFzc2Vzc2FibGVfYmFzZXM9MTE2MjU3MQpyZXRyaWV2YWxfcGFkZGluZ19icD0xMDAwCnN0YXR1cz1GUk9aRU5fQkVGT1JFX1BIQVNFMkNfQ09NUFVURQo=', 'data/manifests/selected_experiments.tsv': 'ZXhwZXJpbWVudF9pZAlzb3VyY2VfY29ycHVzCXNvdXJjZV9zdWJtaXNzaW9uX2lkCXNhbXBsZV9pZAl0ZWNobm9sb2d5CXBsYXRmb3JtCWNvdmVyYWdlCXBpcGVsaW5lX25hbWUJYWxpZ25lcl9uYW1lcwlhbGlnbmVyX3ZlcnNpb25zCWNhbGxlcl9uYW1lcwljYWxsZXJfdmVyc2lvbnMJcXVlcnlfYXJ0aWZhY3RfaWQJc2VsZWN0aW9uX2Jhc2lzCXBlcmZvcm1hbmNlX2NvbnN1bHRlZF9mb3Jfc2VsZWN0aW9uCW1ldGFkYXRhX3N0YXR1cwpleHAtcGZkYS12Mi02MHo1OS1oZzAwMglwcmVjaXNpb25GREEgVHJ1dGggQ2hhbGxlbmdlIFYyIC8gTklTVCBQRFIgbWRzMi0yMzM2CTYwWjU5CUhHMDAyCUlMTFVNSU5BCU5vdmFTZXEgNjAwMAkzNXgJUk4tSWxsdW1pbmEJQldBCTAuNy4xNQlOZXVTb21hdGljO0dBVEs7UGljYXJkCVVOS05PV047NC4xLjcuMDsyLjIyLjQJYXJ0aWZhY3QtcXVlcnktNjB6NTktaGcwMDIJUGhhc2UgMEIgaGFkIGFscmVhZHkgdmVyaWZpZWQgdGhlIGV4YWN0IHB1YmxpYyBCR1pGIGFydGlmYWN0IGhlYWRlciBjaGVja3N1bSBzaW5nbGUtdGVjaG5vbG9neSBtYXBwaW5nIGFuZCBtZXRob2RzOyBjaG9zZW4gYmVmb3JlIHNjb3JlIGluc3BlY3Rpb24JZmFsc2UJUEFSVElBTF9FWEFDVF9DQUxMRVJfVkVSU0lPTl9VTktOT1dOCmV4cC1wZmRhLXYyLXJ1ODhuLWhnMDAyCXByZWNpc2lvbkZEQSBUcnV0aCBDaGFsbGVuZ2UgVjIgLyBOSVNUIFBEUiBtZHMyLTIzMzYJUlU4OE4JSEcwMDIJT05UCVByb21ldGhJT04gUjkuNAk0N3gJUEVQUEVSLURlZXBWYXJpYW50IChPTlQpCW1pbmltYXAyCTIuMTctcjk0MQlQRVBQRVI7RGVlcFZhcmlhbnQJVU5LTk9XTjtVTktOT1dOCWFydGlmYWN0LXF1ZXJ5LXJ1ODhuLWhnMDAyCVVuYW1iaWd1b3VzIHB1YmxpYyBzaW5nbGUtdGVjaG5vbG9neSBPTlQgYXJ0aWZhY3QgYW5kIHBhcnRpY2lwYW50LXN1cHBsaWVkIG1ldGhvZHM7IGNob3NlbiBiZWZvcmUgc2NvcmUgaW5zcGVjdGlvbglmYWxzZQlQQVJUSUFMX0NBTExFUl9WRVJTSU9OU19VTktOT1dOCg==', 'schemas/event.schema.json': 'ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL2V2ZW50LnNjaGVtYS5qc29uIiwKICAidGl0bGUiOiAiRVZFTlQiLAogICJkZXNjcmlwdGlvbiI6ICJBIHJ1bi1zY29wZWQgY29tcGFyaXNvbiBldmVudC4gSXQgaXMgbm90IGEgdW5pdmVyc2FsIGJpb2xvZ2ljYWwtdmFyaWFudCBpZGVudGl0eS4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWwogICAgInNjaGVtYV92ZXJzaW9uIiwKICAgICJldmVudF9pZCIsCiAgICAiaWRlbnRpdHlfc2NvcGUiLAogICAgImJlbmNobWFya19ydW5faWQiLAogICAgInJlZmVyZW5jZV9hcnRpZmFjdF9pZCIsCiAgICAiYXNzZW1ibHkiLAogICAgImNvbnRpZyIsCiAgICAic3RhcnRfMGJhc2VkIiwKICAgICJlbmRfMGJhc2VkIiwKICAgICJkZXJpdmF0aW9uX21ldGhvZCIsCiAgICAiY3Jvc3NfcnVuX2VxdWl2YWxlbmNlX2Fzc2VydGVkIgogIF0sCiAgInByb3BlcnRpZXMiOiB7CiAgICAic2NoZW1hX3ZlcnNpb24iOiB7ImNvbnN0IjogIjEuMC4wIn0sCiAgICAiZXZlbnRfaWQiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgImlkZW50aXR5X3Njb3BlIjogewogICAgICAiZW51bSI6IFsiQ09NUEFSQVRPUl9TVVBFUkxPQ1VTIiwgIlNPVVJDRV9SRUNPUkRfRkFMTEJBQ0siLCAiTk9STUFMSVpFRF9BTExFTEVfTE9PS1VQIl0KICAgIH0sCiAgICAiYmVuY2htYXJrX3J1bl9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAicmVmZXJlbmNlX2FydGlmYWN0X2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJhc3NlbWJseSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiY29udGlnIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzdGFydF8wYmFzZWQiOiB7InR5cGUiOiAiaW50ZWdlciIsICJtaW5pbXVtIjogMH0sCiAgICAiZW5kXzBiYXNlZCI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAxfSwKICAgICJzb3VyY2VfZXZlbnRfaWQiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgIm5vcm1hbGl6ZWRfYWxsZWxlX2xvb2t1cF9rZXkiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgImRlcml2YXRpb25fbWV0aG9kIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJkZXJpdmF0aW9uX3ZlcnNpb24iOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgImNyb3NzX3J1bl9lcXVpdmFsZW5jZV9hc3NlcnRlZCI6IHsiY29uc3QiOiBmYWxzZX0sCiAgICAibm90ZXMiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19CiAgfSwKICAiYWxsT2YiOiBbCiAgICB7CiAgICAgICJpZiI6IHsicHJvcGVydGllcyI6IHsiaWRlbnRpdHlfc2NvcGUiOiB7ImNvbnN0IjogIkNPTVBBUkFUT1JfU1VQRVJMT0NVUyJ9fX0sCiAgICAgICJ0aGVuIjogewogICAgICAgICJyZXF1aXJlZCI6IFsic291cmNlX2V2ZW50X2lkIl0sCiAgICAgICAgInByb3BlcnRpZXMiOiB7InNvdXJjZV9ldmVudF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX19CiAgICAgIH0KICAgIH0KICBdCn0K', 'schemas/observation.schema.json': 'ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL29ic2VydmF0aW9uLnNjaGVtYS5qc29uIiwKICAidGl0bGUiOiAiT0JTRVJWQVRJT04iLAogICJkZXNjcmlwdGlvbiI6ICJPbmUgc2lkZS1zcGVjaWZpYyBzb3VyY2UtcmVjb3JkIGRlY2lzaW9uIHdpdGhpbiBvbmUgcnVuLXNjb3BlZCBldmVudC4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWwogICAgInNjaGVtYV92ZXJzaW9uIiwKICAgICJvYnNlcnZhdGlvbl9pZCIsCiAgICAiYmVuY2htYXJrX3J1bl9pZCIsCiAgICAiZXZlbnRfaWQiLAogICAgInNpZGUiLAogICAgIm9ic2VydmF0aW9uX29yaWdpbiIsCiAgICAicmF3X2RlY2lzaW9uIiwKICAgICJub3JtYWxpemVkX2RlY2lzaW9uIiwKICAgICJyYXdfbWF0Y2hfa2luZCIsCiAgICAicmVnaW9uX3N0YXR1cyIsCiAgICAic291cmNlX291dHB1dF9hcnRpZmFjdF9pZCIsCiAgICAic291cmNlX3JlY29yZF9vcmRpbmFsIiwKICAgICJzb3VyY2VfY29udGlnIiwKICAgICJzb3VyY2VfcG9zXzFiYXNlZCIsCiAgICAic291cmNlX3JlZiIsCiAgICAic291cmNlX2FsdCIsCiAgICAic291cmNlX2ZpbHRlciIsCiAgICAibm9ybWFsaXphdGlvbl9tZXRob2QiLAogICAgIm5vcm1hbGl6YXRpb25fdmVyc2lvbiIKICBdLAogICJwcm9wZXJ0aWVzIjogewogICAgInNjaGVtYV92ZXJzaW9uIjogeyJjb25zdCI6ICIxLjAuMCJ9LAogICAgIm9ic2VydmF0aW9uX2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJiZW5jaG1hcmtfcnVuX2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJldmVudF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAic2lkZSI6IHsiZW51bSI6IFsiVFJVVEgiLCAiUVVFUlkiXX0sCiAgICAib2JzZXJ2YXRpb25fb3JpZ2luIjogeyJjb25zdCI6ICJSRUdFTkVSQVRFRCJ9LAogICAgInJhd19kZWNpc2lvbiI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAibm9ybWFsaXplZF9kZWNpc2lvbiI6IHsiZW51bSI6IFsiVFAiLCAiRlAiLCAiRk4iLCAiTiIsICJVTksiLCAiSUdOIiwgIk9USEVSIl19LAogICAgInJhd19tYXRjaF9raW5kIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJyYXdfZW5naW5lX2RldGFpbCI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0sCiAgICAicmF3X3ZhcmlhbnRfdHlwZSI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0sCiAgICAicmF3X2xvY2F0aW9uX3R5cGUiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgInJlZ2lvbl9zdGF0dXMiOiB7ImVudW0iOiBbIklOU0lERSIsICJPVVRTSURFIiwgIkJPUkRFUiIsICJVTktOT1dOIl19LAogICAgInNvdXJjZV9vdXRwdXRfYXJ0aWZhY3RfaWQiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgInNvdXJjZV9yZWNvcmRfb3JkaW5hbCI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAxfSwKICAgICJzb3VyY2VfY29udGlnIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzb3VyY2VfcG9zXzFiYXNlZCI6IHsidHlwZSI6ICJpbnRlZ2VyIiwgIm1pbmltdW0iOiAxfSwKICAgICJzb3VyY2VfcmVmIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzb3VyY2VfYWx0IjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzb3VyY2VfZ2Vub3R5cGUiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19LAogICAgInNvdXJjZV9maWx0ZXIiOiB7InR5cGUiOiAiYXJyYXkiLCAiaXRlbXMiOiB7InR5cGUiOiAic3RyaW5nIn19LAogICAgInF1YWxpdHlfc2NvcmUiOiB7InR5cGUiOiBbIm51bWJlciIsICJudWxsIl19LAogICAgIm5vcm1hbGl6YXRpb25fbWV0aG9kIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJub3JtYWxpemF0aW9uX3ZlcnNpb24iOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgIm5vcm1hbGl6ZWRfc3RhcnRfMGJhc2VkIjogeyJ0eXBlIjogWyJpbnRlZ2VyIiwgIm51bGwiXSwgIm1pbmltdW0iOiAwfSwKICAgICJub3JtYWxpemVkX2VuZF8wYmFzZWQiOiB7InR5cGUiOiBbImludGVnZXIiLCAibnVsbCJdLCAibWluaW11bSI6IDF9LAogICAgIm5vcm1hbGl6ZWRfcmVmIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdfSwKICAgICJub3JtYWxpemVkX2FsdCI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0sCiAgICAibm90ZXMiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19CiAgfQp9Cgo=', 'schemas/experiment.schema.json': 'ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL2V4cGVyaW1lbnQuc2NoZW1hLmpzb24iLAogICJ0aXRsZSI6ICJFWFBFUklNRU5UIiwKICAiZGVzY3JpcHRpb24iOiAiT25lIHN1Ym1pdHRlZCBwaXBlbGluZSBhcHBsaWVkIHRvIG9uZSBzYW1wbGUgYW5kIHNlcXVlbmNpbmcgY29uZGl0aW9uLiIsCiAgInR5cGUiOiAib2JqZWN0IiwKICAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBmYWxzZSwKICAicmVxdWlyZWQiOiBbCiAgICAic2NoZW1hX3ZlcnNpb24iLAogICAgImV4cGVyaW1lbnRfaWQiLAogICAgInNvdXJjZV9jb3JwdXMiLAogICAgInNvdXJjZV9zdWJtaXNzaW9uX2lkIiwKICAgICJzYW1wbGVfaWQiLAogICAgInRlY2hub2xvZ3kiLAogICAgInBsYXRmb3JtIiwKICAgICJjb3ZlcmFnZSIsCiAgICAicGlwZWxpbmVfbmFtZSIsCiAgICAiY2FsbGVyX25hbWVzIiwKICAgICJjYWxsZXJfdmVyc2lvbnMiLAogICAgInJlZmVyZW5jZV9hc3NlbWJseSIsCiAgICAibWV0YWRhdGFfYXJ0aWZhY3RfaWRzIgogIF0sCiAgInByb3BlcnRpZXMiOiB7CiAgICAic2NoZW1hX3ZlcnNpb24iOiB7ImNvbnN0IjogIjEuMC4wIn0sCiAgICAiZXhwZXJpbWVudF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAic291cmNlX2NvcnB1cyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAic291cmNlX3N1Ym1pc3Npb25faWQiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgInNhbXBsZV9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAidGVjaG5vbG9neSI6IHsiZW51bSI6IFsiSUxMVU1JTkEiLCAiT05UIiwgIlBBQ0JJT19ISUZJIiwgIk1VTFRJIiwgIlVOS05PV04iXX0sCiAgICAicGxhdGZvcm0iOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgImNvdmVyYWdlIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJsaWJyYXJ5X3ByZXBhcmF0aW9uIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxLCAiZGVmYXVsdCI6ICJVTktOT1dOIn0sCiAgICAiYmFzZWNhbGxlciI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMSwgImRlZmF1bHQiOiAiVU5LTk9XTiJ9LAogICAgImJhc2VjYWxsZXJfdmVyc2lvbiI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMSwgImRlZmF1bHQiOiAiVU5LTk9XTiJ9LAogICAgImFsaWduZXJfbmFtZXMiOiB7InR5cGUiOiAiYXJyYXkiLCAiaXRlbXMiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LCAidW5pcXVlSXRlbXMiOiB0cnVlfSwKICAgICJhbGlnbmVyX3ZlcnNpb25zIjogeyJ0eXBlIjogImFycmF5IiwgIml0ZW1zIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfX0sCiAgICAicGlwZWxpbmVfbmFtZSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAicGlwZWxpbmVfdmVyc2lvbiI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMSwgImRlZmF1bHQiOiAiVU5LTk9XTiJ9LAogICAgImNhbGxlcl9uYW1lcyI6IHsidHlwZSI6ICJhcnJheSIsICJtaW5JdGVtcyI6IDEsICJpdGVtcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX19LAogICAgImNhbGxlcl92ZXJzaW9ucyI6IHsidHlwZSI6ICJhcnJheSIsICJtaW5JdGVtcyI6IDEsICJpdGVtcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX19LAogICAgInJlZmVyZW5jZV9hc3NlbWJseSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAibWV0YWRhdGFfYXJ0aWZhY3RfaWRzIjogewogICAgICAidHlwZSI6ICJhcnJheSIsCiAgICAgICJtaW5JdGVtcyI6IDEsCiAgICAgICJ1bmlxdWVJdGVtcyI6IHRydWUsCiAgICAgICJpdGVtcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0KICAgIH0sCiAgICAic2VsZWN0aW9uX2Jhc2lzIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJwZXJmb3JtYW5jZV9jb25zdWx0ZWRfZm9yX3NlbGVjdGlvbiI6IHsiY29uc3QiOiBmYWxzZX0sCiAgICAibm90ZXMiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl19CiAgfQp9Cgo=', 'schemas/source_artifact.schema.json': 'ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL3NvdXJjZV9hcnRpZmFjdC5zY2hlbWEuanNvbiIsCiAgInRpdGxlIjogIlNPVVJDRV9BUlRJRkFDVCIsCiAgImRlc2NyaXB0aW9uIjogIkEgYnl0ZS1hZGRyZXNzYWJsZSBpbnB1dCwgbWV0YWRhdGEsIHNvZnR3YXJlLCBvciBnZW5lcmF0ZWQgb3V0cHV0IG9iamVjdC4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWwogICAgInNjaGVtYV92ZXJzaW9uIiwKICAgICJzb3VyY2VfYXJ0aWZhY3RfaWQiLAogICAgImFydGlmYWN0X25hbWUiLAogICAgImFydGlmYWN0X3JvbGUiLAogICAgInVyaSIsCiAgICAibWVkaWFfdHlwZSIsCiAgICAiYnl0ZV9zaXplIiwKICAgICJ2ZXJzaW9uIiwKICAgICJjaGVja3N1bXMiLAogICAgImNoZWNrc3VtX3N0YXR1cyIsCiAgICAicmV0cmlldmFsX3N0YXR1cyIsCiAgICAibGljZW5zZV9leHByZXNzaW9uIiwKICAgICJyZXVzZV9iYXNpcyIsCiAgICAiZGVyaXZhdGlvbl9zdGF0ZSIKICBdLAogICJwcm9wZXJ0aWVzIjogewogICAgInNjaGVtYV92ZXJzaW9uIjogeyJjb25zdCI6ICIxLjAuMCJ9LAogICAgInNvdXJjZV9hcnRpZmFjdF9pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiYXJ0aWZhY3RfbmFtZSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiYXJ0aWZhY3Rfcm9sZSI6IHsKICAgICAgImVudW0iOiBbCiAgICAgICAgIlFVRVJZX1ZDRiIsICJUUlVUSF9WQ0YiLCAiVFJVVEhfSU5ERVgiLCAiQkVOQ0hNQVJLX0JFRCIsCiAgICAgICAgIlJFRkVSRU5DRV9GQVNUQSIsICJSRUZFUkVOQ0VfSU5ERVgiLCAiRVhQRVJJTUVOVF9NRVRBREFUQSIsCiAgICAgICAgIkRBVEFTRVRfTUFOSUZFU1QiLCAiU09GVFdBUkVfSU1BR0UiLCAiU09GVFdBUkVfQVJDSElWRSIsCiAgICAgICAgIkRFUklWRURfU0xJQ0UiLCAiQ09NUEFSQVRPUl9PVVRQVVQiLCAiUlVOX0xPRyIsICJQQVJRVUVUIiwKICAgICAgICAiRFVDS0RCIiwgIlZBTElEQVRJT05fUkVQT1JUIiwgIk9USEVSIgogICAgICBdCiAgICB9LAogICAgInVyaSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAibWVkaWFfdHlwZSI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiYnl0ZV9zaXplIjogeyJ0eXBlIjogWyJpbnRlZ2VyIiwgIm51bGwiXSwgIm1pbmltdW0iOiAwfSwKICAgICJ2ZXJzaW9uIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJjaGVja3N1bXMiOiB7CiAgICAgICJ0eXBlIjogImFycmF5IiwKICAgICAgIml0ZW1zIjogewogICAgICAgICJ0eXBlIjogIm9iamVjdCIsCiAgICAgICAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgICAgICAgInJlcXVpcmVkIjogWyJhbGdvcml0aG0iLCAidmFsdWUiLCAic291cmNlIl0sCiAgICAgICAgInByb3BlcnRpZXMiOiB7CiAgICAgICAgICAiYWxnb3JpdGhtIjogeyJlbnVtIjogWyJzaGEyNTYiLCAibWQ1IiwgIm9jaS1tYW5pZmVzdC1kaWdlc3QiXX0sCiAgICAgICAgICAidmFsdWUiOiB7InR5cGUiOiAic3RyaW5nIiwgInBhdHRlcm4iOiAiXlswLTlhLWZdezMyLDY0fSQifSwKICAgICAgICAgICJzb3VyY2UiOiB7ImVudW0iOiBbIlBVQkxJU0hFUiIsICJDT01QVVRFRF9PTl9SRVRSSUVWQUwiLCAiUkVHSVNUUlkiXX0KICAgICAgICB9CiAgICAgIH0KICAgIH0sCiAgICAiY2hlY2tzdW1fc3RhdHVzIjogeyJlbnVtIjogWyJQVUJMSVNIRVJfVkVSSUZJRUQiLCAiQ09NUFVURURfTk9UX1lFVF9WRVJJRklFRCIsICJOT1RfWUVUX1JFVFJJRVZFRCJdfSwKICAgICJyZXRyaWV2YWxfc3RhdHVzIjogeyJlbnVtIjogWyJOT1RfRE9XTkxPQURFRCIsICJET1dOTE9BREVEIiwgIkdFTkVSQVRFRCJdfSwKICAgICJyZXRyaWV2ZWRfYXQiOiB7InR5cGUiOiBbInN0cmluZyIsICJudWxsIl0sICJmb3JtYXQiOiAiZGF0ZS10aW1lIn0sCiAgICAibGljZW5zZV9leHByZXNzaW9uIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJyZXVzZV9iYXNpcyI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAiZGVyaXZhdGlvbl9zdGF0ZSI6IHsiZW51bSI6IFsiU09VUkNFIiwgIlRSQU5TRk9STUVEIiwgIkdFTkVSQVRFRCJdfSwKICAgICJwdWJsaXNoZXJfbGFzdF9tb2RpZmllZCI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXSwgImZvcm1hdCI6ICJkYXRlLXRpbWUifSwKICAgICJub3RlcyI6IHsidHlwZSI6IFsic3RyaW5nIiwgIm51bGwiXX0KICB9LAogICJhbGxPZiI6IFsKICAgIHsKICAgICAgImlmIjogeyJwcm9wZXJ0aWVzIjogeyJjaGVja3N1bV9zdGF0dXMiOiB7ImNvbnN0IjogIlBVQkxJU0hFUl9WRVJJRklFRCJ9fX0sCiAgICAgICJ0aGVuIjogeyJwcm9wZXJ0aWVzIjogeyJjaGVja3N1bXMiOiB7Im1pbkl0ZW1zIjogMX19fQogICAgfSwKICAgIHsKICAgICAgImlmIjogeyJwcm9wZXJ0aWVzIjogeyJyZXRyaWV2YWxfc3RhdHVzIjogeyJlbnVtIjogWyJET1dOTE9BREVEIiwgIkdFTkVSQVRFRCJdfX19LAogICAgICAidGhlbiI6IHsKICAgICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAgICJjaGVja3N1bXMiOiB7Im1pbkl0ZW1zIjogMX0sCiAgICAgICAgICAicmV0cmlldmVkX2F0IjogeyJ0eXBlIjogInN0cmluZyJ9CiAgICAgICAgfQogICAgICB9CiAgICB9CiAgXQp9Cgo=', 'schemas/provenance_link.schema.json': 'ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YxL3Byb3ZlbmFuY2VfbGluay5zY2hlbWEuanNvbiIsCiAgInRpdGxlIjogIlBST1ZFTkFOQ0VfTElOSyIsCiAgImRlc2NyaXB0aW9uIjogIkEgZGlyZWN0ZWQgbGluZWFnZSBlZGdlLiBUaGlzIHNjaGVtYSBkb2VzIG5vdCBlbmNvZGUgYmlvbG9naWNhbCBlcXVpdmFsZW5jZS4iLAogICJ0eXBlIjogIm9iamVjdCIsCiAgImFkZGl0aW9uYWxQcm9wZXJ0aWVzIjogZmFsc2UsCiAgInJlcXVpcmVkIjogWwogICAgInNjaGVtYV92ZXJzaW9uIiwKICAgICJwcm92ZW5hbmNlX2xpbmtfaWQiLAogICAgInNvdXJjZV9lbnRpdHlfdHlwZSIsCiAgICAic291cmNlX2VudGl0eV9pZCIsCiAgICAidGFyZ2V0X2VudGl0eV90eXBlIiwKICAgICJ0YXJnZXRfZW50aXR5X2lkIiwKICAgICJyZWxhdGlvbiIsCiAgICAiYXNzZXJ0ZWRfYXQiLAogICAgImFzc2VydGVkX2J5IiwKICAgICJ0cmFuc2Zvcm1hdGlvbiIKICBdLAogICJwcm9wZXJ0aWVzIjogewogICAgInNjaGVtYV92ZXJzaW9uIjogeyJjb25zdCI6ICIxLjAuMCJ9LAogICAgInByb3ZlbmFuY2VfbGlua19pZCI6IHsidHlwZSI6ICJzdHJpbmciLCAibWluTGVuZ3RoIjogMX0sCiAgICAic291cmNlX2VudGl0eV90eXBlIjogeyIkcmVmIjogIiMvJGRlZnMvZW50aXR5X3R5cGUifSwKICAgICJzb3VyY2VfZW50aXR5X2lkIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJ0YXJnZXRfZW50aXR5X3R5cGUiOiB7IiRyZWYiOiAiIy8kZGVmcy9lbnRpdHlfdHlwZSJ9LAogICAgInRhcmdldF9lbnRpdHlfaWQiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgInJlbGF0aW9uIjogewogICAgICAiZW51bSI6IFsiVVNFRCIsICJHRU5FUkFURUQiLCAiREVSSVZFRF9GUk9NIiwgIk5PUk1BTElaRURfRlJPTSIsICJERVNDUklCRVMiLCAiSEFTX01FTUJFUiJdCiAgICB9LAogICAgImFzc2VydGVkX2F0IjogeyJ0eXBlIjogInN0cmluZyIsICJmb3JtYXQiOiAiZGF0ZS10aW1lIn0sCiAgICAiYXNzZXJ0ZWRfYnkiOiB7InR5cGUiOiAic3RyaW5nIiwgIm1pbkxlbmd0aCI6IDF9LAogICAgInRyYW5zZm9ybWF0aW9uIjogeyJ0eXBlIjogInN0cmluZyIsICJtaW5MZW5ndGgiOiAxfSwKICAgICJzb2Z0d2FyZV9uYW1lIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdfSwKICAgICJzb2Z0d2FyZV92ZXJzaW9uIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdfSwKICAgICJwYXJhbWV0ZXJzIjogeyJ0eXBlIjogIm9iamVjdCJ9LAogICAgIm5vdGVzIjogeyJ0eXBlIjogWyJzdHJpbmciLCAibnVsbCJdfQogIH0sCiAgIiRkZWZzIjogewogICAgImVudGl0eV90eXBlIjogewogICAgICAiZW51bSI6IFsiRVZFTlQiLCAiRVhQRVJJTUVOVCIsICJCRU5DSE1BUktfUlVOIiwgIk9CU0VSVkFUSU9OIiwgIlNPVVJDRV9BUlRJRkFDVCIsICJQUk9WRU5BTkNFX0xJTksiXQogICAgfQogIH0KfQoK', 'schemas/phase2/benchmark_run.schema.json': 'ewogICIkc2NoZW1hIjogImh0dHBzOi8vanNvbi1zY2hlbWEub3JnL2RyYWZ0LzIwMjAtMTIvc2NoZW1hIiwKICAiJGlkIjogImh0dHBzOi8vb21pY3NlZGdlYmlvLmV4YW1wbGUvc2NoZW1hcy9wcm9qZWN0MDAzL3YyL2JlbmNobWFya19ydW4uc2NoZW1hLmpzb24iLAogICJ0aXRsZSI6ICJCRU5DSE1BUktfUlVOIFx1MjAxNCBQaGFzZSAyIG11bHRpLXJlZ2lvbiIsCiAgImRlc2NyaXB0aW9uIjogIk9uZSBpbW11dGFibGUgY29tcGFyYXRvciBpbnZvY2F0aW9uIG92ZXIgYSBmcm96ZW4gbXVsdGktcmVnaW9uIGJlbmNobWFyayBwYW5lbC4gVGhpcyBzY2hlbWEgZXh0ZW5kcyB0aGUgUGhhc2UgMSBzaW5nbGUtaW50ZXJ2YWwgcnVuIG1vZGVsIHdpdGhvdXQgbW9kaWZ5aW5nIHRoZSBmcm96ZW4gUGhhc2UgMSBzY2hlbWEuIiwKICAidHlwZSI6ICJvYmplY3QiLAogICJhZGRpdGlvbmFsUHJvcGVydGllcyI6IGZhbHNlLAogICJyZXF1aXJlZCI6IFsKICAgICJzY2hlbWFfdmVyc2lvbiIsCiAgICAiYmVuY2htYXJrX3J1bl9pZCIsCiAgICAiZXhwZXJpbWVudF9pZCIsCiAgICAib2JzZXJ2YXRpb25fb3JpZ2luIiwKICAgICJxdWVyeV9hcnRpZmFjdF9pZCIsCiAgICAidHJ1dGhfYXJ0aWZhY3RfaWQiLAogICAgImJlbmNobWFya19yZWdpb25fc2V0X2FydGlmYWN0X2lkIiwKICAgICJiZW5jaG1hcmtfcmVnaW9uX3NldF9zaGEyNTYiLAogICAgInJldHJpZXZhbF9yZWdpb25fc2V0X2FydGlmYWN0X2lkIiwKICAgICJyZXRyaWV2YWxfcmVnaW9uX3NldF9zaGEyNTYiLAogICAgInJlZmVyZW5jZV9hcnRpZmFjdF9pZCIsCiAgICAicmVmZXJlbmNlX2Fzc2VtYmx5IiwKICAgICJiZW5jaG1hcmtfcmVnaW9uX3NldCIsCiAgICAicmV0cmlldmFsX3BhZGRpbmdfYnAiLAogICAgImNvbXBhcmF0b3IiLAogICAgImNvbXBhcmF0b3JfdmVyc2lvbiIsCiAgICAiZW5naW5lIiwKICAgICJlbmdpbmVfdmVyc2lvbiIsCiAgICAiZW52aXJvbm1lbnRfbG9ja19pZCIsCiAgICAicnVuX3N0YXR1cyIsCiAgICAiY29tbWFuZF9hcmd2IiwKICAgICJvdXRwdXRfYXJ0aWZhY3RfaWRzIgogIF0sCiAgInByb3BlcnRpZXMiOiB7CiAgICAic2NoZW1hX3ZlcnNpb24iOiB7CiAgICAgICJjb25zdCI6ICIyLjAuMCIKICAgIH0sCiAgICAiYmVuY2htYXJrX3J1bl9pZCI6IHsKICAgICAgInR5cGUiOiAic3RyaW5nIiwKICAgICAgIm1pbkxlbmd0aCI6IDEKICAgIH0sCiAgICAiZXhwZXJpbWVudF9pZCI6IHsKICAgICAgInR5cGUiOiAic3RyaW5nIiwKICAgICAgIm1pbkxlbmd0aCI6IDEKICAgIH0sCiAgICAib2JzZXJ2YXRpb25fb3JpZ2luIjogewogICAgICAiY29uc3QiOiAiUkVHRU5FUkFURUQiCiAgICB9LAogICAgInF1ZXJ5X2FydGlmYWN0X2lkIjogewogICAgICAidHlwZSI6ICJzdHJpbmciLAogICAgICAibWluTGVuZ3RoIjogMQogICAgfSwKICAgICJ0cnV0aF9hcnRpZmFjdF9pZCI6IHsKICAgICAgInR5cGUiOiAic3RyaW5nIiwKICAgICAgIm1pbkxlbmd0aCI6IDEKICAgIH0sCiAgICAiYmVuY2htYXJrX3JlZ2lvbl9zZXRfYXJ0aWZhY3RfaWQiOiB7CiAgICAgICJ0eXBlIjogInN0cmluZyIsCiAgICAgICJtaW5MZW5ndGgiOiAxCiAgICB9LAogICAgImJlbmNobWFya19yZWdpb25fc2V0X3NoYTI1NiI6IHsKICAgICAgInR5cGUiOiAic3RyaW5nIiwKICAgICAgInBhdHRlcm4iOiAiXlswLTlhLWZdezY0fSQiCiAgICB9LAogICAgInJldHJpZXZhbF9yZWdpb25fc2V0X2FydGlmYWN0X2lkIjogewogICAgICAidHlwZSI6ICJzdHJpbmciLAogICAgICAibWluTGVuZ3RoIjogMQogICAgfSwKICAgICJyZXRyaWV2YWxfcmVnaW9uX3NldF9zaGEyNTYiOiB7CiAgICAgICJ0eXBlIjogInN0cmluZyIsCiAgICAgICJwYXR0ZXJuIjogIl5bMC05YS1mXXs2NH0kIgogICAgfSwKICAgICJyZWZlcmVuY2VfYXJ0aWZhY3RfaWQiOiB7CiAgICAgICJ0eXBlIjogInN0cmluZyIsCiAgICAgICJtaW5MZW5ndGgiOiAxCiAgICB9LAogICAgInJlZmVyZW5jZV9hc3NlbWJseSI6IHsKICAgICAgImNvbnN0IjogIkdSQ2gzOCIKICAgIH0sCiAgICAiYmVuY2htYXJrX3JlZ2lvbl9zZXQiOiB7CiAgICAgICIkcmVmIjogIiMvJGRlZnMvcmVnaW9uX3NldCIKICAgIH0sCiAgICAicmV0cmlldmFsX3BhZGRpbmdfYnAiOiB7CiAgICAgICJ0eXBlIjogImludGVnZXIiLAogICAgICAibWluaW11bSI6IDAKICAgIH0sCiAgICAiY29tcGFyYXRvciI6IHsKICAgICAgInR5cGUiOiAic3RyaW5nIiwKICAgICAgIm1pbkxlbmd0aCI6IDEKICAgIH0sCiAgICAiY29tcGFyYXRvcl92ZXJzaW9uIjogewogICAgICAidHlwZSI6ICJzdHJpbmciLAogICAgICAibWluTGVuZ3RoIjogMQogICAgfSwKICAgICJlbmdpbmUiOiB7CiAgICAgICJ0eXBlIjogInN0cmluZyIsCiAgICAgICJtaW5MZW5ndGgiOiAxCiAgICB9LAogICAgImVuZ2luZV92ZXJzaW9uIjogewogICAgICAidHlwZSI6ICJzdHJpbmciLAogICAgICAibWluTGVuZ3RoIjogMQogICAgfSwKICAgICJlbnZpcm9ubWVudF9sb2NrX2lkIjogewogICAgICAidHlwZSI6ICJzdHJpbmciLAogICAgICAibWluTGVuZ3RoIjogMQogICAgfSwKICAgICJydW50aW1lX2ltYWdlX2RpZ2VzdCI6IHsKICAgICAgInR5cGUiOiBbCiAgICAgICAgInN0cmluZyIsCiAgICAgICAgIm51bGwiCiAgICAgIF0sCiAgICAgICJwYXR0ZXJuIjogIl5zaGEyNTY6WzAtOWEtZl17NjR9JCIKICAgIH0sCiAgICAiY29tbWFuZF9hcmd2IjogewogICAgICAidHlwZSI6ICJhcnJheSIsCiAgICAgICJpdGVtcyI6IHsKICAgICAgICAidHlwZSI6ICJzdHJpbmciCiAgICAgIH0KICAgIH0sCiAgICAicGFyYW1ldGVycyI6IHsKICAgICAgInR5cGUiOiAib2JqZWN0IgogICAgfSwKICAgICJydW5fc3RhdHVzIjogewogICAgICAiZW51bSI6IFsKICAgICAgICAiUExBTk5FRCIsCiAgICAgICAgIlJVTk5JTkciLAogICAgICAgICJTVUNDRUVERUQiLAogICAgICAgICJGQUlMRUQiCiAgICAgIF0KICAgIH0sCiAgICAic3RhcnRlZF9hdCI6IHsKICAgICAgInR5cGUiOiBbCiAgICAgICAgInN0cmluZyIsCiAgICAgICAgIm51bGwiCiAgICAgIF0sCiAgICAgICJmb3JtYXQiOiAiZGF0ZS10aW1lIgogICAgfSwKICAgICJjb21wbGV0ZWRfYXQiOiB7CiAgICAgICJ0eXBlIjogWwogICAgICAgICJzdHJpbmciLAogICAgICAgICJudWxsIgogICAgICBdLAogICAgICAiZm9ybWF0IjogImRhdGUtdGltZSIKICAgIH0sCiAgICAib3V0cHV0X2FydGlmYWN0X2lkcyI6IHsKICAgICAgInR5cGUiOiAiYXJyYXkiLAogICAgICAidW5pcXVlSXRlbXMiOiB0cnVlLAogICAgICAiaXRlbXMiOiB7CiAgICAgICAgInR5cGUiOiAic3RyaW5nIiwKICAgICAgICAibWluTGVuZ3RoIjogMQogICAgICB9CiAgICB9LAogICAgImZhaWx1cmVfcmVhc29uIjogewogICAgICAidHlwZSI6IFsKICAgICAgICAic3RyaW5nIiwKICAgICAgICAibnVsbCIKICAgICAgXQogICAgfSwKICAgICJub3RlcyI6IHsKICAgICAgInR5cGUiOiBbCiAgICAgICAgInN0cmluZyIsCiAgICAgICAgIm51bGwiCiAgICAgIF0KICAgIH0KICB9LAogICIkZGVmcyI6IHsKICAgICJyZWdpb25fc2V0IjogewogICAgICAidHlwZSI6ICJvYmplY3QiLAogICAgICAiYWRkaXRpb25hbFByb3BlcnRpZXMiOiBmYWxzZSwKICAgICAgInJlcXVpcmVkIjogWwogICAgICAgICJhc3NlbWJseSIsCiAgICAgICAgImNvb3JkaW5hdGVfc3lzdGVtIiwKICAgICAgICAic2VsZWN0ZWRfd2luZG93X2NvdW50IiwKICAgICAgICAic2VnbWVudF9jb3VudCIsCiAgICAgICAgImNvbnRleHRfY291bnQiLAogICAgICAgICJjaHJvbW9zb21lX2NvdW50IiwKICAgICAgICAiY2hyb21vc29tZXMiLAogICAgICAgICJleGNsdWRlZF9jaHJvbW9zb21lcyIsCiAgICAgICAgImFzc2Vzc2FibGVfYmFzZXMiCiAgICAgIF0sCiAgICAgICJwcm9wZXJ0aWVzIjogewogICAgICAgICJhc3NlbWJseSI6IHsKICAgICAgICAgICJjb25zdCI6ICJHUkNoMzgiCiAgICAgICAgfSwKICAgICAgICAiY29vcmRpbmF0ZV9zeXN0ZW0iOiB7CiAgICAgICAgICAiY29uc3QiOiAiMC1iYXNlZCBoYWxmLW9wZW4iCiAgICAgICAgfSwKICAgICAgICAic2VsZWN0ZWRfd2luZG93X2NvdW50IjogewogICAgICAgICAgInR5cGUiOiAiaW50ZWdlciIsCiAgICAgICAgICAibWluaW11bSI6IDEKICAgICAgICB9LAogICAgICAgICJzZWdtZW50X2NvdW50IjogewogICAgICAgICAgInR5cGUiOiAiaW50ZWdlciIsCiAgICAgICAgICAibWluaW11bSI6IDEKICAgICAgICB9LAogICAgICAgICJjb250ZXh0X2NvdW50IjogewogICAgICAgICAgInR5cGUiOiAiaW50ZWdlciIsCiAgICAgICAgICAibWluaW11bSI6IDEKICAgICAgICB9LAogICAgICAgICJjaHJvbW9zb21lX2NvdW50IjogewogICAgICAgICAgInR5cGUiOiAiaW50ZWdlciIsCiAgICAgICAgICAibWluaW11bSI6IDEKICAgICAgICB9LAogICAgICAgICJjaHJvbW9zb21lcyI6IHsKICAgICAgICAgICJ0eXBlIjogImFycmF5IiwKICAgICAgICAgICJtaW5JdGVtcyI6IDEsCiAgICAgICAgICAidW5pcXVlSXRlbXMiOiB0cnVlLAogICAgICAgICAgIml0ZW1zIjogewogICAgICAgICAgICAidHlwZSI6ICJzdHJpbmciLAogICAgICAgICAgICAibWluTGVuZ3RoIjogMQogICAgICAgICAgfQogICAgICAgIH0sCiAgICAgICAgImV4Y2x1ZGVkX2Nocm9tb3NvbWVzIjogewogICAgICAgICAgInR5cGUiOiAiYXJyYXkiLAogICAgICAgICAgInVuaXF1ZUl0ZW1zIjogdHJ1ZSwKICAgICAgICAgICJpdGVtcyI6IHsKICAgICAgICAgICAgInR5cGUiOiAic3RyaW5nIiwKICAgICAgICAgICAgIm1pbkxlbmd0aCI6IDEKICAgICAgICAgIH0KICAgICAgICB9LAogICAgICAgICJhc3Nlc3NhYmxlX2Jhc2VzIjogewogICAgICAgICAgInR5cGUiOiAiaW50ZWdlciIsCiAgICAgICAgICAibWluaW11bSI6IDEKICAgICAgICB9CiAgICAgIH0KICAgIH0KICB9LAogICJhbGxPZiI6IFsKICAgIHsKICAgICAgImlmIjogewogICAgICAgICJwcm9wZXJ0aWVzIjogewogICAgICAgICAgInJ1bl9zdGF0dXMiOiB7CiAgICAgICAgICAgICJjb25zdCI6ICJTVUNDRUVERUQiCiAgICAgICAgICB9CiAgICAgICAgfQogICAgICB9LAogICAgICAidGhlbiI6IHsKICAgICAgICAicHJvcGVydGllcyI6IHsKICAgICAgICAgICJjb21tYW5kX2FyZ3YiOiB7CiAgICAgICAgICAgICJtaW5JdGVtcyI6IDEKICAgICAgICAgIH0sCiAgICAgICAgICAib3V0cHV0X2FydGlmYWN0X2lkcyI6IHsKICAgICAgICAgICAgIm1pbkl0ZW1zIjogMQogICAgICAgICAgfSwKICAgICAgICAgICJydW50aW1lX2ltYWdlX2RpZ2VzdCI6IHsKICAgICAgICAgICAgInR5cGUiOiAic3RyaW5nIgogICAgICAgICAgfSwKICAgICAgICAgICJzdGFydGVkX2F0IjogewogICAgICAgICAgICAidHlwZSI6ICJzdHJpbmciCiAgICAgICAgICB9LAogICAgICAgICAgImNvbXBsZXRlZF9hdCI6IHsKICAgICAgICAgICAgInR5cGUiOiAic3RyaW5nIgogICAgICAgICAgfQogICAgICAgIH0KICAgICAgfQogICAgfQogIF0KfQo='}
EMBEDDED_SHA256 = {'environment/comparator-lock.yaml': '7d5d907d4c9f26dcc41f78325da3b35506a2562689d8b3d240343957f16f250a', 'data/phase2/admitted_sources.tsv': '55ebf63901ac0c118cb5ec687b49cc81cf680bb9357246ba2f6138b27a29641f', 'data/phase2/selected_assessable_segments.bed': '4c77e17c3dec47051cb9b29fa14fee67f31a559785417d083f4f7bcacb05cd26', 'data/phase2/selected_intervals.tsv': '2e5a6a97c9bacf5fdf3cf547ac7db32d06f611a1f2bcca66cbcfbd04e722c01c', 'data/phase2/interval_panel.lock': 'ef00c21f0e667c1891b90d6785f1072895a8b9d620a5519130a8a1823a4bd9a1', 'docs/phase2/phase2_protocol.md': '6234485e8916fd30a335c761a62668b617cb148b5b5a1753d12fd7765fa1633f', 'docs/phase2/phase2c_compute_protocol.md': 'e2eb2bb92c89b1497a10d0f0b6d57b91b95c20b79193e9e0fd255403027961be', 'docs/phase2/phase2c_compute_protocol.lock': '4a1ff34bbcc15e609dda7a02895ce45c6804f880364b4c2e09921840332f1ba3', 'docs/phase2/benchmark_run_schema_extension.lock': 'ed1dcd409236dd21429ac9ff70f065b8560145eefb6c6d32e042e597c5d80434', 'data/manifests/selected_experiments.tsv': '26c5fc1d7dcd257aec705952af8261221e626e73f66680107126221f7e4e4420', 'schemas/event.schema.json': 'fe0c2ef9782a9bb876875c477057326fb973af8e3618fa0368cfdadee2cd721b', 'schemas/observation.schema.json': '4577760d99d80188eb9486ab829afc1d72356a61e34557daaf7b41504e701c5e', 'schemas/experiment.schema.json': '9989cfc91722d9423fc215bd08e96c05c11a7b91d64e87899909283292a70f04', 'schemas/source_artifact.schema.json': '224906a07723e680e69ca77636c71508c81ecd51c74a918452c5dc888e11d9f1', 'schemas/provenance_link.schema.json': '2458fbdfd3a7cef5104775ebb545f32f0cb74aa83d9a3ca660af0773335ca818', 'schemas/phase2/benchmark_run.schema.json': '2db3310a8f52dc30f0668b5cf0bd25fb91b1bed35f8c1111191b80d38410c2d7'}

for rel, payload in EMBEDDED_B64.items():
    raw = base64.b64decode(payload)
    actual = hashlib.sha256(raw).hexdigest()
    expected = EMBEDDED_SHA256[rel]
    if actual != expected:
        raise RuntimeError(f"embedded checksum mismatch: {rel}")
    target = CONFIG_DIR / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(raw)

def read_tsv(path):
    with open(path, newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle, delimiter="\t"))

admitted = read_tsv(CONFIG_DIR / "data/phase2/admitted_sources.tsv")
if len(admitted) != 4:
    raise RuntimeError("Expected exactly four admitted Phase 2 sources")

expected_candidates = {
    "phase2-hg003-60z59",
    "phase2-hg003-ru88n",
    "phase2-hg004-60z59",
    "phase2-hg004-ru88n",
}
observed_candidates = {row["candidate_id"] for row in admitted}
if observed_candidates != expected_candidates:
    raise RuntimeError("Frozen Phase 2 admitted-source set changed")

if any(
    row["performance_consulted_for_selection"].lower() != "false"
    for row in admitted
):
    raise RuntimeError("Outcome-independent source selection violated")

if any(row["phase2a_admission_state"] != "ELIGIBLE" for row in admitted):
    raise RuntimeError("A non-eligible Phase 2 source is present")

panel_path = CONFIG_DIR / "data/phase2/selected_assessable_segments.bed"
panel = []
for raw in panel_path.read_text(encoding="utf-8").splitlines():
    if raw.strip():
        fields = raw.split("\t")
        panel.append((fields[0], int(fields[1]), int(fields[2])))

panel_bases = sum(end - start for _, start, end in panel)
chromosomes = sorted({chrom for chrom, _, _ in panel})

if len(panel) != 541:
    raise RuntimeError(f"Expected 541 panel segments, found {len(panel)}")
if panel_bases != 1162571:
    raise RuntimeError(
        f"Expected 1162571 assessable bases, found {panel_bases}"
    )
if len(chromosomes) != 19:
    raise RuntimeError(f"Expected 19 chromosomes, found {len(chromosomes)}")
if "chr20" in chromosomes:
    raise RuntimeError("chr20 is forbidden in Phase 2C")

print("FROZEN_PHASE2C_CONFIGURATION_PASS")
print(json.dumps({
    "runs": len(admitted),
    "segments": len(panel),
    "assessable_bases": panel_bases,
    "chromosomes": chromosomes,
    "embedded_files": len(EMBEDDED_B64),
}, indent=2))


## 3. Materialize the audited Phase 2C runner

The runner is embedded in this notebook at generation time. Its SHA-256
must match the locally audited runner before it can be written or executed.


In [ ]:
RUNNER_B64 = '...'
EXPECTED_RUNNER_SHA256 = 'eb1838e100e31cadbae052cc1f08c175208e9e6046ef4365efa36d4ba0b4a718'

runner_bytes = base64.b64decode(RUNNER_B64)
runner_sha = hashlib.sha256(runner_bytes).hexdigest()
if runner_sha != EXPECTED_RUNNER_SHA256:
    raise RuntimeError(
        "Embedded Phase 2C runner checksum mismatch: "
        f"{runner_sha} != {EXPECTED_RUNNER_SHA256}"
    )

RUNNER = RUNNER_DIR / "run_phase2c_colab.py"
RUNNER.write_bytes(runner_bytes)
RUNNER.chmod(0o755)

compile(
    RUNNER.read_text(encoding="utf-8"),
    str(RUNNER),
    "exec",
)

print("RUNNER_MATERIALIZATION_PASS")
print("runner:", RUNNER)
print("runner SHA-256:", runner_sha)


## 4. Execute Phase 2C

This is the only cell that starts package installation, public genomic
downloads, hap.py/vcfeval, evidence materialization, and result packaging.

Leave `EXECUTE_COMPARISON = False` until the complete Phase 2C pre-compute
package has been committed and pushed.


In [ ]:
if not EXECUTE_COMPARISON:
    raise RuntimeError(
        "EXECUTE_COMPARISON is False. The frozen Phase 2C pre-compute "
        "package must be committed and pushed before genomic compute."
    )

completed = subprocess.run(
    [sys.executable, str(RUNNER)],
    text=True,
)

if completed.returncode != 0:
    raise RuntimeError(
        f"Phase 2C runner failed with exit code {completed.returncode}"
    )

if not ARCHIVE_PATH.exists():
    raise RuntimeError(
        "Phase 2C runner completed without producing the compact archive"
    )

archive_sha = hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
print("PHASE2C_RUNNER_EXECUTION_PASS")
print("archive:", ARCHIVE_PATH)
print("archive size:", ARCHIVE_PATH.stat().st_size)
print("archive SHA-256:", archive_sha)


## 5. Download compact result archive

Run only after the execution cell finishes with `PHASE2C_COMPUTE_PASS` and
`PHASE2C_RUNNER_EXECUTION_PASS`.


In [ ]:
if not EXECUTE_COMPARISON:
    raise RuntimeError("Phase 2C compute has not been enabled")
if not ARCHIVE_PATH.exists():
    raise RuntimeError("Phase 2C compact result archive is absent")

from google.colab import files
files.download(str(ARCHIVE_PATH))


## Scientific boundary

Phase 2C creates run-scoped benchmark evidence only. It does not merge
events across runs or assert biological identity. Phase 2D applies the
frozen biological identity layer, and Phase 2E evaluates the final
20-criterion Phase 2 gate.
